## warp installation

In [ ]:
!pip install warp-lang

## Base code

Unoptimized baseline implementation using isolated kernels and an Array of Structures (AoS) memory layout.

In [ ]:
%%writefile FDM_WARP_TEST.py
import sys
import numpy as np
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# Initialize Warp
# wp.clear_kernel_cache()
#wp.init()
# Select GPU if available, else fallback to CPU backend for Warp
#device = "cuda" if wp.is_cuda_available() else "cpu"

wp.init()

print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

if wp.is_cuda_available():
    wp.set_device("cuda:1")
    device = "cuda:1"
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100 # Your grid size

TOTAL_RUNS = 7
IS_PROFILING = False

# Parse Slurm arguments
if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells

Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

# ========== SCHEME SELECTION ==========
NUMERICAL_SCHEME = 'muscl'  # or 'central'
TIME_STEPPING = 'euler'  # or 'rk2'

# ========== PERFORMANCE OPTIMIZATIONS ==========
SHOW_LIVE_PLOTS = False  # Keep False to avoid live plotting overhead
PRINT_EVERY_N_STEPS = 20  # Print progress

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================

U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

# Convert to (Nx, Ny, 4) layout for Warp's vec4f
U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS & FUNCTIONS
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    u = U[1] / rho
    v = U[2] / rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    uL = UL[1] / rhoL
    vL = UL[2] / rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    uR = UR[1] / rhoR
    vR = UR[2] / rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL / rhoL)
    cR = wp.sqrt(gamma * pR / rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) / (rhoL + rhoR))

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) / (SR - SL)

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i = i
    src_j = j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

@wp.kernel
def compute_fluxes_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float, use_muscl: int, Nx: int, Ny: int):
    i, j = wp.tid()

    # X-flux at interface (i+1/2, j)
    if i >= 1 and i <= Nx - 3 and j >= 1 and j <= Ny - 2:
        if use_muscl == 1:
            UL_x, UR_x = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        else:
            UL_x = U[i, j]
            UR_x = U[i+1, j]
        Fx[i, j] = hllc_flux(UL_x, UR_x, gamma, EPS, 0)

    # Y-flux at interface (i, j+1/2)
    if i >= 1 and i <= Nx - 2 and j >= 1 and j <= Ny - 3:
        if use_muscl == 1:
            UL_y, UR_y = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        else:
            UL_y = U[i, j]
            UR_y = U[i, j+1]
        Fy[i, j] = hllc_flux(UL_y, UR_y, gamma, EPS, 1)

@wp.kernel
def compute_rhs_kernel(Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), dx: float, dy: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rhs[i, j] = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy

@wp.kernel
def euler_step_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def rk2_step1_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U1[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def rk2_step2_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), rhs1: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = 0.5 * (U[i, j] + U1[i, j] + rhs1[i, j] * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================

def get_numpy_state(U_wp):
    # Retrieve from GPU and transpose back to (4, Nx, Ny)
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    """Create comprehensive final plots after simulation"""

    # Extract fields
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)  # Avoid negative pressure

    # Mach number
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    # Schlieren (gradient of density)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    # Create physical coordinates
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Plot 1: Density
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    # Plot 2: Pressure
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    # Plot 3: Velocity magnitude
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    # Plot 4: Mach Number
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    # Plot 5: Schlieren
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    # Plot 6: Streamlines or velocity vectors (simplified)
    # Downsample for clarity
    stride = 10
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6,
                     extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {NUMERICAL_SCHEME.upper()} Scheme, t={t_final:.3f}')
    plt.tight_layout()
    plt.savefig('final_results_updated1.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Also create a line plot at mid-plane for comparison
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure')
    plt.xlabel('x')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('midplane_profiles_updated1.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n✓ Final plots saved as 'final_results.png' and 'midplane_profiles.png'")

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

''''''


# ============================================
# MAIN GPU SIMULATION LOOP NEW
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

# Calculate fixed dt based on initial conditions
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()  # Important: synchronize before reading from GPU
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps needed: {int(t_final/dt_fixed) + 1}")
print("="*50)

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    # CRITICAL: Reset the initial state for every single run
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0

    wp.synchronize()
    start_time = time.time()

    # Run simulation with fixed dt
    while t < t_final:
        if IS_PROFILING and step >= 3:
              break
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        # Use fixed dt, adjust only the last step
        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk2':
            # RK2 Step 1
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

            # RK2 Step 2 (SSPRK2)
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

        # Only print step progress during the last run to avoid log spam
        if run == TOTAL_RUNS - 1 and step % PRINT_EVERY_N_STEPS == 0:
            wp.synchronize()
            elapsed_loop = time.time() - start_time
            steps_per_sec = step / elapsed_loop if elapsed_loop > 0 else 0
            U_np = get_numpy_state(U_wp)
            rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
            actual_cfl = initial_max_speed * dt / min(dx, dy)
            print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.6f} | "
                  f"CFL={actual_cfl:.3f} | ρ={rho_mean:.4f} | p={p_mean:.4f} | "
                  f"speed={steps_per_sec:.1f} steps/s")

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    # Drop the first run (warm-up) if we did multiple runs
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    # Calculate TFLOPS based on the best time
    flops_per_cell_per_step = 2000 if TIME_STEPPING == 'rk2' else 1000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling complete. Ignore execution time due to Nsight Compute overhead.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

# Get final state
U_final = get_numpy_state(U_wp)

# Create comprehensive final plots
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

# Create validation plot
plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('validation_updated1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Validation plot saved as 'validation.png'")
print("\nAll plots generated successfully!")


## opti1

Reduces computational overhead by eliminating expensive division operations

In [ ]:
%%writefile FDM_WARP_TEST_opti_1.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # ADD THIS LINE HERE
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# Initialize Warp
# wp.clear_kernel_cache()
#wp.init()
# Select GPU if available, else fallback to CPU backend for Warp
#device = "cuda" if wp.is_cuda_available() else "cpu"

wp.init()

wp.init()

# 1. Set the device FIRST
if wp.is_cuda_available():
    wp.set_device("cuda:2")
    device = "cuda:2"  # <-- Explicitly assign 'cuda:3' to the variable
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# 2. Print the device AFTER setting it
print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100 # Your grid size

TOTAL_RUNS = 7
IS_PROFILING = False

# Parse Slurm arguments
if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells

Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

# ========== SCHEME SELECTION ==========
NUMERICAL_SCHEME = 'muscl'  # or 'central'
TIME_STEPPING = 'euler'  # or 'rk2'

# ========== PERFORMANCE OPTIMIZATIONS ==========
SHOW_LIVE_PLOTS = False  # Keep False to avoid live plotting overhead
PRINT_EVERY_N_STEPS = 20  # Print progress

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================

U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

# Convert to (Nx, Ny, 4) layout for Warp's vec4f
U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS & FUNCTIONS
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho  # Calculate inverse once
    u = U[1] * inv_rho   # Multiply instead of divide
    #u = U[1] / rho
    v = U[2] * inv_rho
    #v  = U[2] / rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    # Avoid another division by multiplying the inverse of the sum
    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i = i
    src_j = j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

@wp.kernel
def compute_fluxes_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float, use_muscl: int, Nx: int, Ny: int):
    i, j = wp.tid()

    # X-flux at interface (i+1/2, j)
    if i >= 1 and i <= Nx - 3 and j >= 1 and j <= Ny - 2:
        if use_muscl == 1:
            UL_x, UR_x = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        else:
            UL_x = U[i, j]
            UR_x = U[i+1, j]
        Fx[i, j] = hllc_flux(UL_x, UR_x, gamma, EPS, 0)

    # Y-flux at interface (i, j+1/2)
    if i >= 1 and i <= Nx - 2 and j >= 1 and j <= Ny - 3:
        if use_muscl == 1:
            UL_y, UR_y = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        else:
            UL_y = U[i, j]
            UR_y = U[i, j+1]
        Fy[i, j] = hllc_flux(UL_y, UR_y, gamma, EPS, 1)

@wp.kernel
def compute_rhs_kernel(Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), dx: float, dy: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rhs[i, j] = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy

@wp.kernel
def euler_step_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def euler_fused_step_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2),
    Fx: wp.array(dtype=wp.vec4f, ndim=2),
    Fy: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2),
    dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Calculate RHS strictly in registers, never saving to global memory
        local_rhs = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy
        U_new[i, j] = U[i, j] + local_rhs * dt


@wp.kernel
def rk2_step1_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U1[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def rk2_step2_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), rhs1: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = 0.5 * (U[i, j] + U1[i, j] + rhs1[i, j] * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================

def get_numpy_state(U_wp):
    # Retrieve from GPU and transpose back to (4, Nx, Ny)
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    """Generate descriptive filename for plots based on scheme and grid size"""
    return f"{base_name}_{scheme}_{"opti1"}_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    """Create comprehensive final plots after simulation"""

    # Generate filename prefix with grid info
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    # Extract fields
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)  # Avoid negative pressure

    # Mach number
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    # Schlieren (gradient of density)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    # Create physical coordinates
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Plot 1: Density
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    # Plot 2: Pressure
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    # Plot 3: Velocity magnitude
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    # Plot 4: Mach Number
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    # Plot 5: Schlieren
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    # Plot 6: Streamlines or velocity vectors (simplified)
    # Downsample for clarity
    stride = max(1, min(NX, NY) // 50)  # Adaptive stride based on grid size
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6,
                     extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme, t={t_final:.3f}')
    plt.tight_layout()

    # Save with descriptive filename
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")
    #plt.show()

    # Also create a line plot at mid-plane for comparison
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure')
    plt.xlabel('x')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    # Save midplane plot with descriptive filename
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")
    #plt.show()

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

'''print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

wp.synchronize()
start_time = time.time()
t = 0.0
step = 0

estimated_steps = int(t_final / (CFL * min(dx, dy) / 0.5))
print(f"Estimated total steps: ~{estimated_steps}")
print("="*50)


# Run simulation without live plotting
while t < t_final:
    wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

    # Calculate dt dynamically via atomic reduction
    max_speed_arr.fill_(0.0)
    wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
    max_speed = max_speed_arr.numpy()[0]
    dt = CFL * min(dx, dy) / max_speed
    if t + dt > t_final: dt = t_final - t

    if TIME_STEPPING == 'euler':
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    elif TIME_STEPPING == 'rk2':
        # RK2 Step 1
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

        # RK2 Step 2 (SSPRK2)
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    t += dt
    step += 1

    if step % PRINT_EVERY_N_STEPS == 0:
        elapsed = time.time() - start_time
        steps_per_sec = step / elapsed if elapsed > 0 else 0
        U_np = get_numpy_state(U_wp)
        rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
        print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.5f} | "
              f"ρ={rho_mean:.4f} | p={p_mean:.4f} | speed={steps_per_sec:.1f} steps/s")

wp.synchronize()
end_time = time.time()
print("="*50)
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print(f"Total steps: {step}")
print(f"Average speed: {step/(end_time-start_time):.1f} steps/second")


# ============================================
# MAIN GPU SIMULATION LOOP NEW
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

wp.synchronize()
start_time = time.time()
t = 0.0
step = 0

# Calculate fixed dt based on initial conditions
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()  # Important: synchronize before reading from GPU
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps needed: {int(t_final/dt_fixed) + 1}")
print("="*50)

# Run simulation with fixed dt
while t < t_final:
    wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

    # Use fixed dt, adjust only the last step
    dt = dt_fixed
    if t + dt > t_final:
        dt = t_final - t

    if TIME_STEPPING == 'euler':
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    elif TIME_STEPPING == 'rk2':
        # RK2 Step 1
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

        # RK2 Step 2 (SSPRK2)
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    t += dt
    step += 1

    if step % PRINT_EVERY_N_STEPS == 0:
        wp.synchronize()  # Synchronize for accurate timing
        elapsed = time.time() - start_time
        steps_per_sec = step / elapsed if elapsed > 0 else 0
        U_np = get_numpy_state(U_wp)
        rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
        # Calculate actual CFL for monitoring
        actual_cfl = initial_max_speed * dt / min(dx, dy)
        print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.6f} | "
              f"CFL={actual_cfl:.3f} | ρ={rho_mean:.4f} | p={p_mean:.4f} | "
              f"speed={steps_per_sec:.1f} steps/s")

wp.synchronize()
end_time = time.time()
print("="*50)
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print(f"Total steps: {step}")
print(f"Average speed: {step/(end_time-start_time):.1f} steps/second")'''


# ============================================
# MAIN GPU SIMULATION LOOP NEW
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

# Calculate fixed dt based on initial conditions
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()  # Important: synchronize before reading from GPU
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps needed: {int(t_final/dt_fixed) + 1}")
print("="*50)

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    # CRITICAL: Reset the initial state for every single run
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0

    wp.synchronize()
    start_time = time.time()

    # Run simulation with fixed dt
    while t < t_final:
        if IS_PROFILING and step >= 3:
              break
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        # Use fixed dt, adjust only the last step
        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        '''if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)'''

        if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            # Launch the fused kernel instead of the two separate ones
            wp.launch(euler_fused_step_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, U_new_wp, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk2':
            # RK2 Step 1
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

            # RK2 Step 2 (SSPRK2)
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

        # Only print step progress during the last run to avoid log spam
        if run == TOTAL_RUNS - 1 and step % PRINT_EVERY_N_STEPS == 0:
            wp.synchronize()
            elapsed_loop = time.time() - start_time
            steps_per_sec = step / elapsed_loop if elapsed_loop > 0 else 0
            U_np = get_numpy_state(U_wp)
            rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
            actual_cfl = initial_max_speed * dt / min(dx, dy)
            print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.6f} | "
                  f"CFL={actual_cfl:.3f} | ρ={rho_mean:.4f} | p={p_mean:.4f} | "
                  f"speed={steps_per_sec:.1f} steps/s")

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    # Drop the first run (warm-up) if we did multiple runs
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    # Calculate TFLOPS based on the best time
    flops_per_cell_per_step = 2000 if TIME_STEPPING == 'rk2' else 1000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling complete. Ignore execution time due to Nsight Compute overhead.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

# Get final state
U_final = get_numpy_state(U_wp)

# Create comprehensive final plots
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

# Create validation plot
plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

validation_filename = generate_plot_filename('valdiation', NUMERICAL_SCHEME, grid_size = (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print("\n✓ Validation plot saved as 'validation.png'")
print("\nAll plots generated successfully!")



## opti2

Explores primitive variable passing to reduce redundant state conversions.

In [ ]:
%%writefile FDM_WARP_TEST_opti_2.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # ADD THIS LINE HERE
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# Initialize Warp
# wp.clear_kernel_cache()
#wp.init()
# Select GPU if available, else fallback to CPU backend for Warp
#device = "cuda" if wp.is_cuda_available() else "cpu"

wp.init()

wp.init()

# 1. Set the device FIRST
if wp.is_cuda_available():
    wp.set_device("cuda:2")
    device = "cuda:2"  # <-- Explicitly assign 'cuda:3' to the variable
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# 2. Print the device AFTER setting it
print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100 # Your grid size

TOTAL_RUNS = 7
IS_PROFILING = False

# Parse Slurm arguments
if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells

Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

# ========== SCHEME SELECTION ==========
NUMERICAL_SCHEME = 'muscl'  # or 'central'
TIME_STEPPING = 'euler'  # or 'rk2'

# ========== PERFORMANCE OPTIMIZATIONS ==========
SHOW_LIVE_PLOTS = False  # Keep False to avoid live plotting overhead
PRINT_EVERY_N_STEPS = 20  # Print progress

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================

U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

# Convert to (Nx, Ny, 4) layout for Warp's vec4f
U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS & FUNCTIONS
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho  # Calculate inverse once
    u = U[1] * inv_rho   # Multiply instead of divide
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR
'''
@wp.func
def reconstruct_state_prim(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    # Pack PRIMITIVE states into vec4f to pass them easily
    Prim_L = wp.vec4f(rho_L, u_L, v_L, p_L)
    Prim_R = wp.vec4f(rho_R, u_R, v_R, p_R)
    return Prim_L, Prim_R
'''
@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    # Avoid another division by multiplying the inverse of the sum
    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux
'''
@wp.func
def hllc_flux_prim(Prim_L: wp.vec4f, Prim_R: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    # Unpack Primitives
    rhoL = wp.max(Prim_L[0], EPS)
    uL = Prim_L[1]
    vL = Prim_L[2]
    pL = wp.max(Prim_L[3], EPS)

    rhoR = wp.max(Prim_R[0], EPS)
    uR = Prim_R[1]
    vR = Prim_R[2]
    pR = wp.max(Prim_R[3], EPS)

    # Swap velocities if checking Y direction
    if dir_y == 1:
        uL, vL = vL, uL
        uR, vR = vR, uR

    # Calculate Total Energy once for the final flux vector
    EL = pL / (gamma - 1.0) + 0.5 * rhoL * (uL * uL + vL * vL)
    ER = pR / (gamma - 1.0) + 0.5 * rhoR * (uR * uR + vR * vR)

    inv_rhoL = 1.0 / rhoL
    inv_rhoR = 1.0 / rhoR

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (EL + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (ER + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        # Convert Primitives back to Conservative ONLY for the star region calculation
        UL_cons = wp.vec4f(rhoL, rhoL * uL, rhoL * vL, EL)
        UR_cons = wp.vec4f(rhoR, rhoR * uR, rhoR * vR, ER)
        flux = (SR * FL - SL * FR + SL * SR * (UR_cons - UL_cons)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux
    '''

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i = i
    src_j = j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

@wp.kernel
def compute_fluxes_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float, use_muscl: int, Nx: int, Ny: int):
    i, j = wp.tid()

    # X-flux at interface (i+1/2, j)
    if i >= 1 and i <= Nx - 3 and j >= 1 and j <= Ny - 2:
        if use_muscl == 1:
            UL_x, UR_x = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        else:
            UL_x = U[i, j]
            UR_x = U[i+1, j]
        Fx[i, j] = hllc_flux(UL_x, UR_x, gamma, EPS, 0)

    # Y-flux at interface (i, j+1/2)
    if i >= 1 and i <= Nx - 2 and j >= 1 and j <= Ny - 3:
        if use_muscl == 1:
            UL_y, UR_y = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        else:
            UL_y = U[i, j]
            UR_y = U[i, j+1]
        Fy[i, j] = hllc_flux(UL_y, UR_y, gamma, EPS, 1)
'''
@wp.kernel
def compute_fluxes_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float, use_muscl: int, Nx: int, Ny: int):
    i, j = wp.tid()

    # X-flux at interface (i+1/2, j)
    if i >= 1 and i <= Nx - 3 and j >= 1 and j <= Ny - 2:
        if use_muscl == 1:
            Prim_L_x, Prim_R_x = reconstruct_state_prim(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        else:
            rho1, u1, v1, p1 = cons_to_prim(U[i, j], gamma, EPS)
            rho2, u2, v2, p2 = cons_to_prim(U[i+1, j], gamma, EPS)
            Prim_L_x = wp.vec4f(rho1, u1, v1, p1)
            Prim_R_x = wp.vec4f(rho2, u2, v2, p2)

        Fx[i, j] = hllc_flux_prim(Prim_L_x, Prim_R_x, gamma, EPS, 0)

    # Y-flux at interface (i, j+1/2)
    if i >= 1 and i <= Nx - 2 and j >= 1 and j <= Ny - 3:
        if use_muscl == 1:
            Prim_L_y, Prim_R_y = reconstruct_state_prim(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        else:
            rho1, u1, v1, p1 = cons_to_prim(U[i, j], gamma, EPS)
            rho2, u2, v2, p2 = cons_to_prim(U[i, j+1], gamma, EPS)
            Prim_L_y = wp.vec4f(rho1, u1, v1, p1)
            Prim_R_y = wp.vec4f(rho2, u2, v2, p2)

        Fy[i, j] = hllc_flux_prim(Prim_L_y, Prim_R_y, gamma, EPS, 1)'''

@wp.kernel
def compute_rhs_kernel(Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), dx: float, dy: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rhs[i, j] = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy

@wp.kernel
def euler_step_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def euler_fused_step_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2),
    Fx: wp.array(dtype=wp.vec4f, ndim=2),
    Fy: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2),
    dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Calculate RHS strictly in registers, never saving to global memory
        local_rhs = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy
        U_new[i, j] = U[i, j] + local_rhs * dt


@wp.kernel
def rk2_step1_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), rhs: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U1[i, j] = U[i, j] + rhs[i, j] * dt

@wp.kernel
def rk2_step2_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2), rhs1: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2), dt: float):
    i, j = wp.tid()
    U_new[i, j] = 0.5 * (U[i, j] + U1[i, j] + rhs1[i, j] * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================

def get_numpy_state(U_wp):
    # Retrieve from GPU and transpose back to (4, Nx, Ny)
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    """Generate descriptive filename for plots based on scheme and grid size"""
    return f"{base_name}_{scheme}_opti2_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    """Create comprehensive final plots after simulation"""

    # Generate filename prefix with grid info
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    # Extract fields
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)  # Avoid negative pressure

    # Mach number
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    # Schlieren (gradient of density)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    # Create physical coordinates
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Plot 1: Density
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    # Plot 2: Pressure
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    # Plot 3: Velocity magnitude
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    # Plot 4: Mach Number
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    # Plot 5: Schlieren
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    # Plot 6: Streamlines or velocity vectors (simplified)
    # Downsample for clarity
    stride = max(1, min(NX, NY) // 50)  # Adaptive stride based on grid size
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6,
                     extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme, t={t_final:.3f}')
    plt.tight_layout()

    # Save with descriptive filename
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")
    #plt.show()

    # Also create a line plot at mid-plane for comparison
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure')
    plt.xlabel('x')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    # Save midplane plot with descriptive filename
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")
    #plt.show()
    """Create comprehensive final plots after simulation"""

    # Extract fields
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)  # Avoid negative pressure

    # Mach number
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    # Schlieren (gradient of density)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    # Create physical coordinates
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Plot 1: Density
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    # Plot 2: Pressure
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    # Plot 3: Velocity magnitude
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    # Plot 4: Mach Number
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    # Plot 5: Schlieren
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    # Plot 6: Streamlines or velocity vectors (simplified)
    # Downsample for clarity
    stride = 10
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6,
                     extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {NUMERICAL_SCHEME.upper()} Scheme, t={t_final:.3f}')
    plt.tight_layout()
    plt.savefig('final_results_updated1.png', dpi=300, bbox_inches='tight')
    #plt.show()

    # Also create a line plot at mid-plane for comparison
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure')
    plt.xlabel('x')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('midplane_profiles_updated1.png', dpi=300, bbox_inches='tight')
    #plt.show()

    print("\n✓ Final plots saved as 'final_results.png' and 'midplane_profiles.png'")

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

'''print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

wp.synchronize()
start_time = time.time()
t = 0.0
step = 0

estimated_steps = int(t_final / (CFL * min(dx, dy) / 0.5))
print(f"Estimated total steps: ~{estimated_steps}")
print("="*50)


# Run simulation without live plotting
while t < t_final:
    wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

    # Calculate dt dynamically via atomic reduction
    max_speed_arr.fill_(0.0)
    wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
    max_speed = max_speed_arr.numpy()[0]
    dt = CFL * min(dx, dy) / max_speed
    if t + dt > t_final: dt = t_final - t

    if TIME_STEPPING == 'euler':
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    elif TIME_STEPPING == 'rk2':
        # RK2 Step 1
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

        # RK2 Step 2 (SSPRK2)
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    t += dt
    step += 1

    if step % PRINT_EVERY_N_STEPS == 0:
        elapsed = time.time() - start_time
        steps_per_sec = step / elapsed if elapsed > 0 else 0
        U_np = get_numpy_state(U_wp)
        rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
        print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.5f} | "
              f"ρ={rho_mean:.4f} | p={p_mean:.4f} | speed={steps_per_sec:.1f} steps/s")

wp.synchronize()
end_time = time.time()
print("="*50)
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print(f"Total steps: {step}")
print(f"Average speed: {step/(end_time-start_time):.1f} steps/second")


# ============================================
# MAIN GPU SIMULATION LOOP NEW
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

wp.synchronize()
start_time = time.time()
t = 0.0
step = 0

# Calculate fixed dt based on initial conditions
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()  # Important: synchronize before reading from GPU
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps needed: {int(t_final/dt_fixed) + 1}")
print("="*50)

# Run simulation with fixed dt
while t < t_final:
    wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

    # Use fixed dt, adjust only the last step
    dt = dt_fixed
    if t + dt > t_final:
        dt = t_final - t

    if TIME_STEPPING == 'euler':
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    elif TIME_STEPPING == 'rk2':
        # RK2 Step 1
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

        # RK2 Step 2 (SSPRK2)
        wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
        wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
        wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
        wp.copy(U_wp, U_new_wp)

    t += dt
    step += 1

    if step % PRINT_EVERY_N_STEPS == 0:
        wp.synchronize()  # Synchronize for accurate timing
        elapsed = time.time() - start_time
        steps_per_sec = step / elapsed if elapsed > 0 else 0
        U_np = get_numpy_state(U_wp)
        rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
        # Calculate actual CFL for monitoring
        actual_cfl = initial_max_speed * dt / min(dx, dy)
        print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.6f} | "
              f"CFL={actual_cfl:.3f} | ρ={rho_mean:.4f} | p={p_mean:.4f} | "
              f"speed={steps_per_sec:.1f} steps/s")

wp.synchronize()
end_time = time.time()
print("="*50)
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print(f"Total steps: {step}")
print(f"Average speed: {step/(end_time-start_time):.1f} steps/second")'''


# ============================================
# MAIN GPU SIMULATION LOOP NEW
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate Warp Arrays
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
rhs_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

# Calculate fixed dt based on initial conditions
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()  # Important: synchronize before reading from GPU
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps needed: {int(t_final/dt_fixed) + 1}")
print("="*50)

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    # CRITICAL: Reset the initial state for every single run
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0

    wp.synchronize()
    start_time = time.time()

    # Run simulation with fixed dt
    while t < t_final:
        if IS_PROFILING and step >= 3:
              break
        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        # Use fixed dt, adjust only the last step
        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        '''if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(euler_step_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)'''

        if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            # Launch the fused kernel instead of the two separate ones
            wp.launch(euler_fused_step_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, U_new_wp, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk2':
            # RK2 Step 1
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, rhs_wp, U1_wp, float(dt)], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

            # RK2 Step 2 (SSPRK2)
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(compute_rhs_kernel, dim=(Nx, Ny), inputs=[Fx_wp, Fy_wp, rhs_wp, dx, dy, NG, Nx, Ny], device=device)
            wp.launch(rk2_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, rhs_wp, U_new_wp, float(dt)], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

        # Only print step progress during the last run to avoid log spam
        if run == TOTAL_RUNS - 1 and step % PRINT_EVERY_N_STEPS == 0:
            wp.synchronize()
            elapsed_loop = time.time() - start_time
            steps_per_sec = step / elapsed_loop if elapsed_loop > 0 else 0
            U_np = get_numpy_state(U_wp)
            rho_mean, u_mean, p_mean = compute_mean_values_np(U_np)
            actual_cfl = initial_max_speed * dt / min(dx, dy)
            print(f"Step {step:6d} | t={t:.5f}/{t_final} | dt={dt:.6f} | "
                  f"CFL={actual_cfl:.3f} | ρ={rho_mean:.4f} | p={p_mean:.4f} | "
                  f"speed={steps_per_sec:.1f} steps/s")

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    # Drop the first run (warm-up) if we did multiple runs
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    # Calculate TFLOPS based on the best time
    flops_per_cell_per_step = 2000 if TIME_STEPPING == 'rk2' else 1000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling complete. Ignore execution time due to Nsight Compute overhead.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

# Get final state
U_final = get_numpy_state(U_wp)

# Create comprehensive final plots
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

# Create validation plot
plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
validation_filename = generate_plot_filename('valdiation', NUMERICAL_SCHEME, grid_size = (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print("\n✓ Validation plot saved as 'validation.png'")
print("\nAll plots generated successfully!")


Writing FDM_WARP_TEST_opti_2.py


## opti3

Fuses the RHS calculation with the Runge-Kutta time-stepping kernels to reduce global memory roundtrips.

In [ ]:
%%writefile FDM_WARP_TEST_opti3.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # Headless mode for server
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# Initialize Warp
wp.init()

# 1. Set the device FIRST
if wp.is_cuda_available():
    wp.set_device("cuda:1")
    device = "cuda:1"  # <-- Explicitly assign 'cuda:3' to the variable
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# 2. Print the device AFTER setting it
print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100

TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'
TIME_STEPPING = 'euler'

PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================
U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS (OPTIMIZED FAST MATH)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho  # Fast Math Optimization
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

@wp.kernel
def compute_fluxes_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float, use_muscl: int, Nx: int, Ny: int):
    i, j = wp.tid()

    if i >= 1 and i <= Nx - 3 and j >= 1 and j <= Ny - 2:
        if use_muscl == 1:
            UL_x, UR_x = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        else:
            UL_x = U[i, j]
            UR_x = U[i+1, j]
        Fx[i, j] = hllc_flux(UL_x, UR_x, gamma, EPS, 0)

    if i >= 1 and i <= Nx - 2 and j >= 1 and j <= Ny - 3:
        if use_muscl == 1:
            UL_y, UR_y = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        else:
            UL_y = U[i, j]
            UR_y = U[i, j+1]
        Fy[i, j] = hllc_flux(UL_y, UR_y, gamma, EPS, 1)

# --- FUSED EULER KERNEL ---
@wp.kernel
def euler_fused_step_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2),
    Fy: wp.array(dtype=wp.vec4f, ndim=2), U_new: wp.array(dtype=wp.vec4f, ndim=2),
    dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        local_rhs = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy
        U_new[i, j] = U[i, j] + local_rhs * dt

# --- FUSED RK2 KERNELS ---
@wp.kernel
def rk2_fused_step1_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), Fx: wp.array(dtype=wp.vec4f, ndim=2),
    Fy: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rhs = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy
        U1[i, j] = U[i, j] + rhs * dt

@wp.kernel
def rk2_fused_step2_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    Fx: wp.array(dtype=wp.vec4f, ndim=2), Fy: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2), dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rhs1 = (Fx[i-1, j] - Fx[i, j]) / dx + (Fy[i, j-1] - Fy[i, j]) / dy
        U_new[i, j] = 0.5 * (U[i, j] + U1[i, j] + rhs1 * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================
def get_numpy_state(U_wp):
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    """Generate descriptive filename for plots based on scheme and grid size"""
    return f"{base_name}_{scheme}_opti3_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    """Create comprehensive final plots after simulation"""

    # Generate filename prefix with grid info
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    # Extract fields
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)  # Avoid negative pressure

    # Mach number
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    # Schlieren (gradient of density)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    # Create physical coordinates
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    # Create figure with 2x3 subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Plot 1: Density
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    # Plot 2: Pressure
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    # Plot 3: Velocity magnitude
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    # Plot 4: Mach Number
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    # Plot 5: Schlieren
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray',
                           extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    # Plot 6: Streamlines or velocity vectors (simplified)
    # Downsample for clarity
    stride = max(1, min(NX, NY) // 50)  # Adaptive stride based on grid size
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6,
                     extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme, t={t_final:.3f}')
    plt.tight_layout()

    # Save with descriptive filename
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")
    # plt.show() # Disabled for headless execution

    # Also create a line plot at mid-plane for comparison
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure')
    plt.xlabel('x')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    # Save midplane plot with descriptive filename
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")
    # plt.show() # Disabled for headless execution

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fx_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
Fy_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

use_muscl_flag = 1 if NUMERICAL_SCHEME == 'muscl' else 0

max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    wp.synchronize()
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
              break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        if TIME_STEPPING == 'euler':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(euler_fused_step_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, U_new_wp, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk2':
            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(rk2_fused_step1_kernel, dim=(Nx, Ny), inputs=[U_wp, Fx_wp, Fy_wp, U1_wp, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

            wp.launch(compute_fluxes_kernel, dim=(Nx, Ny), inputs=[U1_wp, Fx_wp, Fy_wp, gamma, EPS, use_muscl_flag, Nx, Ny], device=device)
            wp.launch(rk2_fused_step2_kernel, dim=(Nx, Ny), inputs=[U_wp, U1_wp, Fx_wp, Fy_wp, U_new_wp, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    flops_per_cell_per_step = 2000 if TIME_STEPPING == 'rk2' else 1000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling complete.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

# Get final state
U_final = get_numpy_state(U_wp)

# Create comprehensive final plots
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

# Create validation plot
plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"\n✓ Validation plot saved as '{validation_filename}'")
print("\nAll plots generated successfully!")


## opti4

Eliminates intermediate flux arrays by computing the entire spatial operator locally within GPU registers.

In [ ]:
%%writefile FDM_WARP_TEST_opti4.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # Headless mode for server
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
# 1. Turn on Extreme GPU Optimizations
wp.config.verify_fp = False    # Disable NaN/Inf checking for speed
wp.config.fast_math = True     # Enable aggressive compiler math optimizations
wp.init()

# 2. Set the device FIRST
if wp.is_cuda_available():
    wp.set_device("cuda:1")
    device = "cuda:1"  # Explicitly assign 'cuda:3' to the variable
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# 3. Print the device AFTER setting it
print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100

TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'
TIME_STEPPING = 'euler' # or 'rk2'
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================
U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS (EXTREME FUSION)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho  # Fast Math Optimization
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# --- EXTREME FUSED EULER KERNEL ---
@wp.kernel
def extreme_fused_euler_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # X Left Interface
        UL_x1, UR_x1 = reconstruct_state(U[i-2, j], U[i-1, j], U[i, j], U[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)

        # X Right Interface
        UL_x2, UR_x2 = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)

        # Y Bottom Interface
        UL_y1, UR_y1 = reconstruct_state(U[i, j-2], U[i, j-1], U[i, j], U[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)

        # Y Top Interface
        UL_y2, UR_y2 = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        # Update
        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        U_new[i, j] = U[i, j] + local_rhs * dt

# --- EXTREME FUSED RK2 KERNELS ---
@wp.kernel
def extreme_fused_rk2_step1_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        UL_x1, UR_x1 = reconstruct_state(U[i-2, j], U[i-1, j], U[i, j], U[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U[i, j-2], U[i, j-1], U[i, j], U[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        U1[i, j] = U[i, j] + local_rhs * dt

@wp.kernel
def extreme_fused_rk2_step2_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float,
    dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        UL_x1, UR_x1 = reconstruct_state(U1[i-2, j], U1[i-1, j], U1[i, j], U1[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U1[i-1, j], U1[i, j], U1[i+1, j], U1[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U1[i, j-2], U1[i, j-1], U1[i, j], U1[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U1[i, j-1], U1[i, j], U1[i, j+1], U1[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs1 = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        U_new[i, j] = 0.5 * (U[i, j] + U1[i, j] + local_rhs1 * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================
def get_numpy_state(U_wp):
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    """Generate descriptive filename for plots based on scheme and grid size"""
    return f"{base_name}_{scheme}_opti4_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    """Create comprehensive final plots after simulation"""
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)

    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme, t={t_final:.3f}')
    plt.tight_layout()

    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")

    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

    plt.tight_layout()

    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# ---------------------------------------------------------
# VRAM SAVINGS! We no longer allocate Fx_wp and Fy_wp here!
# ---------------------------------------------------------
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
max_speed_arr = wp.zeros(1, dtype=float, device=device)

max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    wp.synchronize()
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
              break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        # Use Block Dim 256 to force better L1 Cache alignment
        if TIME_STEPPING == 'euler':
            wp.launch(extreme_fused_euler_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U_new_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk2':
            wp.launch(extreme_fused_rk2_step1_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U1_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)
            wp.launch(extreme_fused_rk2_step2_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U1_wp, U_new_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    # UPDATE: Extreme Fusion computes 4 fluxes locally instead of reading from memory.
    # Total mathematical workload per cell is significantly higher!
    flops_per_cell_per_step = 8000 if TIME_STEPPING == 'rk2' else 4000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS (Actual Math Workload): {tflops:.6f} TFLOPS")
else:
    print("Profiling complete.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

U_final = get_numpy_state(U_wp)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"\n✓ Validation plot saved as '{validation_filename}'")
print("\nAll plots generated successfully!")


## opti5_rk3

Extends the extreme fusion architecture to support a robust 3-stage SSP-RK3 scheme.

In [ ]:
%%writefile FDM_WARP_TEST_opti5.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # Headless mode for server
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
# 1. Turn on Extreme GPU Optimizations
wp.config.verify_fp = False    # Disable NaN/Inf checking for speed
wp.config.fast_math = True     # Enable aggressive compiler math optimizations
wp.init()

# 2. Set the device FIRST
if wp.is_cuda_available():
    wp.set_device("cuda:2")
    device = "cuda:2"  # Explicitly assign 'cuda:3' to the variable
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

# 3. Print the device AFTER setting it
print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100

TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'
TIME_STEPPING = 'rk3' # <-- Now defaults to RK3
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (NumPy)
# ============================================
U_np = np.zeros((4, Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        U_np[0,i,j] = rho
        U_np[1,i,j] = rho*u
        U_np[2,i,j] = rho*v
        U_np[3,i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

U_initial = np.transpose(U_np, (1, 2, 0)).copy()

# ============================================
# WARP KERNELS (EXTREME FUSION)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho  # Fast Math Optimization
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        U[i, j] = U[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(U: wp.array(dtype=wp.vec4f, ndim=2), max_speed_arr: wp.array(dtype=float), gamma: float, EPS: float, NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        rho, u, v, p = cons_to_prim(U[i, j], gamma, EPS)
        c = wp.sqrt(gamma * p / rho)
        speed = wp.abs(u) + wp.abs(v) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# --- EXTREME FUSED EULER KERNEL ---
@wp.kernel
def extreme_fused_euler_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        UL_x1, UR_x1 = reconstruct_state(U[i-2, j], U[i-1, j], U[i, j], U[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U[i, j-2], U[i, j-1], U[i, j], U[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        U_new[i, j] = U[i, j] + local_rhs * dt

# --- EXTREME FUSED SSP-RK3 KERNELS ---
@wp.kernel
def extreme_fused_rk3_step1_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Evaluate fluxes on U^n
        UL_x1, UR_x1 = reconstruct_state(U[i-2, j], U[i-1, j], U[i, j], U[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U[i-1, j], U[i, j], U[i+1, j], U[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U[i, j-2], U[i, j-1], U[i, j], U[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U[i, j-1], U[i, j], U[i, j+1], U[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        # U^(1) = U^n + dt * L(U^n)
        U1[i, j] = U[i, j] + local_rhs * dt

@wp.kernel
def extreme_fused_rk3_step2_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U1: wp.array(dtype=wp.vec4f, ndim=2),
    U2: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float,
    dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Evaluate fluxes on U^(1)
        UL_x1, UR_x1 = reconstruct_state(U1[i-2, j], U1[i-1, j], U1[i, j], U1[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U1[i-1, j], U1[i, j], U1[i+1, j], U1[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U1[i, j-2], U1[i, j-1], U1[i, j], U1[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U1[i, j-1], U1[i, j], U1[i, j+1], U1[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs1 = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        # U^(2) = 3/4 * U^n + 1/4 * (U^(1) + dt * L(U^(1)))
        U2[i, j] = 0.75 * U[i, j] + 0.25 * (U1[i, j] + local_rhs1 * dt)

@wp.kernel
def extreme_fused_rk3_step3_kernel(
    U: wp.array(dtype=wp.vec4f, ndim=2), U2: wp.array(dtype=wp.vec4f, ndim=2),
    U_new: wp.array(dtype=wp.vec4f, ndim=2), gamma: float, EPS: float,
    dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Evaluate fluxes on U^(2)
        UL_x1, UR_x1 = reconstruct_state(U2[i-2, j], U2[i-1, j], U2[i, j], U2[i+1, j], gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U2[i-1, j], U2[i, j], U2[i+1, j], U2[i+2, j], gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        UL_y1, UR_y1 = reconstruct_state(U2[i, j-2], U2[i, j-1], U2[i, j], U2[i, j+1], gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U2[i, j-1], U2[i, j], U2[i, j+1], U2[i, j+2], gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs2 = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy
        # U^(n+1) = 1/3 * U^n + 2/3 * (U^(2) + dt * L(U^(2)))
        U_new[i, j] = (1.0 / 3.0) * U[i, j] + (2.0 / 3.0) * (U2[i, j] + local_rhs2 * dt)


# ============================================
# UTILITY FUNCTIONS
# ============================================
def get_numpy_state(U_wp):
    return U_wp.numpy().transpose((2, 0, 1))

def compute_mean_values_np(U):
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / rho
    v = U[2, NG:-NG, NG:-NG] / rho
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    return np.mean(rho), np.mean(np.sqrt(u*u + v*v)), np.mean(p)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    return f"{base_name}_{scheme}_opti5_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)

    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme (RK3), t={t_final:.3f}')
    plt.tight_layout()

    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")

    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

    plt.tight_layout()

    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Memory allocation for RK3 stages
U_wp = wp.array(U_initial, dtype=wp.vec4f, device=device)
U_new_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U1_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device)
U2_wp = wp.zeros((Nx, Ny), dtype=wp.vec4f, device=device) # Need one more for RK3
max_speed_arr = wp.zeros(1, dtype=float, device=device)

max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[U_wp, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

run_times = []

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    wp.copy(U_wp, wp.array(U_initial, dtype=wp.vec4f, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    wp.synchronize()
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
              break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U_wp, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        if TIME_STEPPING == 'euler':
            wp.launch(extreme_fused_euler_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U_new_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        elif TIME_STEPPING == 'rk3':
            # RK3 - Stage 1
            wp.launch(extreme_fused_rk3_step1_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U1_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U1_wp, NG, Nx, Ny], device=device)

            # RK3 - Stage 2
            wp.launch(extreme_fused_rk3_step2_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U1_wp, U2_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=[U2_wp, NG, Nx, Ny], device=device)

            # RK3 - Stage 3
            wp.launch(extreme_fused_rk3_step3_kernel, dim=(Nx, Ny), block_dim=256, inputs=[U_wp, U2_wp, U_new_wp, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
            wp.copy(U_wp, U_new_wp)

        t += dt
        step += 1

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    # RK3 evaluates the spatial operator 3 times per step
    flops_per_cell_per_step = 12000 if TIME_STEPPING == 'rk3' else 4000

    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS (Actual Math Workload): {tflops:.6f} TFLOPS")
else:
    print("Profiling complete.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

U_final = get_numpy_state(U_wp)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"\n✓ Validation plot saved as '{validation_filename}'")
print("\nAll plots generated successfully!")


## opti6_soa

Refactors the data architecture from Array of Structures (AoS) to Structure of Arrays (SoA).

In [ ]:
%%writefile FDM_WARP_TEST_opti6.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg') # Headless mode for server
import matplotlib.pyplot as plt
import time
from matplotlib.colors import LogNorm
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
# 1. Turn on Extreme GPU Optimizations
wp.config.verify_fp = False    # Disable NaN/Inf checking for speed
wp.config.fast_math = True     # Enable aggressive compiler math optimizations
wp.init()

# 2. Set the device
if wp.is_cuda_available():
    wp.set_device("cuda:2")
    device = "cuda:2"
else:
    print("!!CUDA NOT AVAILABLE - FALLING BACK TO CPU")
    device = "cpu"

print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100

TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE: Running 1 iteration only ---\n")
else:
    print(f"\n--- BENCHMARK MODE ACTIVE: Running {TOTAL_RUNS} iterations ---\n")

print(f"Initializing grid with size: {NX}x{NY}")

NG = 2  # Ghost cells
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'
TIME_STEPPING = 'rk3'
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (NumPy - SoA Layout)
# ============================================
rho_np = np.zeros((Nx, Ny), dtype=np.float32)
rhou_np = np.zeros((Nx, Ny), dtype=np.float32)
rhov_np = np.zeros((Nx, Ny), dtype=np.float32)
E_np = np.zeros((Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1

        rho_np[i,j] = rho
        rhou_np[i,j] = rho*u
        rhov_np[i,j] = rho*v
        E_np[i,j] = p/(gamma-1.0) + 0.5*rho*(u*u+v*v)

# ============================================
# WARP KERNELS (SOA ARCHITECTURE)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u * u + v * v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u * u + v * v)
    return wp.vec4f(rho, rho * u, rho * v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    if a > 0.0:
        return wp.min(a, b)
    else:
        return wp.max(a, b)

@wp.func
def load_U(rho: wp.array(dtype=wp.float32, ndim=2),
           rhou: wp.array(dtype=wp.float32, ndim=2),
           rhov: wp.array(dtype=wp.float32, ndim=2),
           E: wp.array(dtype=wp.float32, ndim=2),
           i: int, j: int):
    """Helper to load SoA arrays perfectly coalesced into local registers"""
    return wp.vec4f(rho[i,j], rhou[i,j], rhov[i,j], E[i,j])

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f, gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0, gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0 - u_m1, u_p1 - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0 - v_m1, v_p1 - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0 - p_m1, p_p1 - p_0)

    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1 - u_0, u_p2 - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1 - v_0, v_p2 - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1 - p_0, p_p2 - p_p1)

    UL = prim_to_cons(rho_L, u_L, v_L, p_L, gamma)
    UR = prim_to_cons(rho_R, u_R, v_R, p_R, gamma)
    return UL, UR

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL * uL + vL * vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR * uR + vR * vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    flux = wp.vec4f(0.0, 0.0, 0.0, 0.0)
    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])

    return flux

@wp.kernel
def apply_bc_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2),
    rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2),
    E: wp.array(dtype=wp.float32, ndim=2),
    NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    src_i, src_j = i, j

    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1

    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1

    if i != src_i or j != src_j:
        rho[i, j] = rho[src_i, src_j]
        rhou[i, j] = rhou[src_i, src_j]
        rhov[i, j] = rhov[src_i, src_j]
        E[i, j] = E[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2),
    rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2),
    E: wp.array(dtype=wp.float32, ndim=2),
    max_speed_arr: wp.array(dtype=float),
    gamma: float, EPS: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U_i = load_U(rho, rhou, rhov, E, i, j)
        rho_p, u_p, v_p, p_p = cons_to_prim(U_i, gamma, EPS)
        c = wp.sqrt(gamma * p_p / rho_p)
        speed = wp.abs(u_p) + wp.abs(v_p) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# --- EXTREME FUSED EULER KERNEL (SOA) ---
@wp.kernel
def extreme_fused_euler_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2), rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2), E: wp.array(dtype=wp.float32, ndim=2),
    rho_new: wp.array(dtype=wp.float32, ndim=2), rhou_new: wp.array(dtype=wp.float32, ndim=2),
    rhov_new: wp.array(dtype=wp.float32, ndim=2), E_new: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U_im2 = load_U(rho, rhou, rhov, E, i-2, j)
        U_im1 = load_U(rho, rhou, rhov, E, i-1, j)
        U_i   = load_U(rho, rhou, rhov, E, i, j)
        U_ip1 = load_U(rho, rhou, rhov, E, i+1, j)
        U_ip2 = load_U(rho, rhou, rhov, E, i+2, j)

        UL_x1, UR_x1 = reconstruct_state(U_im2, U_im1, U_i, U_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)

        UL_x2, UR_x2 = reconstruct_state(U_im1, U_i, U_ip1, U_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)

        U_jm2 = load_U(rho, rhou, rhov, E, i, j-2)
        U_jm1 = load_U(rho, rhou, rhov, E, i, j-1)
        U_jp1 = load_U(rho, rhou, rhov, E, i, j+1)
        U_jp2 = load_U(rho, rhou, rhov, E, i, j+2)

        UL_y1, UR_y1 = reconstruct_state(U_jm2, U_jm1, U_i, U_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)

        UL_y2, UR_y2 = reconstruct_state(U_jm1, U_i, U_jp1, U_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy

        rho_new[i, j]  = U_i[0] + local_rhs[0] * dt
        rhou_new[i, j] = U_i[1] + local_rhs[1] * dt
        rhov_new[i, j] = U_i[2] + local_rhs[2] * dt
        E_new[i, j]    = U_i[3] + local_rhs[3] * dt

# --- EXTREME FUSED SSP-RK3 KERNELS (SOA) ---
@wp.kernel
def extreme_fused_rk3_step1_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2), rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2), E: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U_im2 = load_U(rho, rhou, rhov, E, i-2, j)
        U_im1 = load_U(rho, rhou, rhov, E, i-1, j)
        U_i   = load_U(rho, rhou, rhov, E, i, j)
        U_ip1 = load_U(rho, rhou, rhov, E, i+1, j)
        U_ip2 = load_U(rho, rhou, rhov, E, i+2, j)

        UL_x1, UR_x1 = reconstruct_state(U_im2, U_im1, U_i, U_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)

        UL_x2, UR_x2 = reconstruct_state(U_im1, U_i, U_ip1, U_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)

        U_jm2 = load_U(rho, rhou, rhov, E, i, j-2)
        U_jm1 = load_U(rho, rhou, rhov, E, i, j-1)
        U_jp1 = load_U(rho, rhou, rhov, E, i, j+1)
        U_jp2 = load_U(rho, rhou, rhov, E, i, j+2)

        UL_y1, UR_y1 = reconstruct_state(U_jm2, U_jm1, U_i, U_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)

        UL_y2, UR_y2 = reconstruct_state(U_jm1, U_i, U_jp1, U_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy

        rho1[i, j]  = U_i[0] + local_rhs[0] * dt
        rhou1[i, j] = U_i[1] + local_rhs[1] * dt
        rhov1[i, j] = U_i[2] + local_rhs[2] * dt
        E1[i, j]    = U_i[3] + local_rhs[3] * dt

@wp.kernel
def extreme_fused_rk3_step2_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U1_im2 = load_U(rho1, rhou1, rhov1, E1, i-2, j)
        U1_im1 = load_U(rho1, rhou1, rhov1, E1, i-1, j)
        U1_i   = load_U(rho1, rhou1, rhov1, E1, i, j)
        U1_ip1 = load_U(rho1, rhou1, rhov1, E1, i+1, j)
        U1_ip2 = load_U(rho1, rhou1, rhov1, E1, i+2, j)

        UL_x1, UR_x1 = reconstruct_state(U1_im2, U1_im1, U1_i, U1_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)

        UL_x2, UR_x2 = reconstruct_state(U1_im1, U1_i, U1_ip1, U1_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)

        U1_jm2 = load_U(rho1, rhou1, rhov1, E1, i, j-2)
        U1_jm1 = load_U(rho1, rhou1, rhov1, E1, i, j-1)
        U1_jp1 = load_U(rho1, rhou1, rhov1, E1, i, j+1)
        U1_jp2 = load_U(rho1, rhou1, rhov1, E1, i, j+2)

        UL_y1, UR_y1 = reconstruct_state(U1_jm2, U1_jm1, U1_i, U1_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)

        UL_y2, UR_y2 = reconstruct_state(U1_jm1, U1_i, U1_jp1, U1_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs1 = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy

        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho2[i, j]  = 0.75 * Un_i[0] + 0.25 * (U1_i[0] + local_rhs1[0] * dt)
        rhou2[i, j] = 0.75 * Un_i[1] + 0.25 * (U1_i[1] + local_rhs1[1] * dt)
        rhov2[i, j] = 0.75 * Un_i[2] + 0.25 * (U1_i[2] + local_rhs1[2] * dt)
        E2[i, j]    = 0.75 * Un_i[3] + 0.25 * (U1_i[3] + local_rhs1[3] * dt)

@wp.kernel
def extreme_fused_rk3_step3_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    rho_new: wp.array(dtype=wp.float32, ndim=2), rhou_new: wp.array(dtype=wp.float32, ndim=2),
    rhov_new: wp.array(dtype=wp.float32, ndim=2), E_new: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float, NG: int, Nx: int, Ny: int
):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U2_im2 = load_U(rho2, rhou2, rhov2, E2, i-2, j)
        U2_im1 = load_U(rho2, rhou2, rhov2, E2, i-1, j)
        U2_i   = load_U(rho2, rhou2, rhov2, E2, i, j)
        U2_ip1 = load_U(rho2, rhou2, rhov2, E2, i+1, j)
        U2_ip2 = load_U(rho2, rhou2, rhov2, E2, i+2, j)

        UL_x1, UR_x1 = reconstruct_state(U2_im2, U2_im1, U2_i, U2_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)

        UL_x2, UR_x2 = reconstruct_state(U2_im1, U2_i, U2_ip1, U2_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)

        U2_jm2 = load_U(rho2, rhou2, rhov2, E2, i, j-2)
        U2_jm1 = load_U(rho2, rhou2, rhov2, E2, i, j-1)
        U2_jp1 = load_U(rho2, rhou2, rhov2, E2, i, j+1)
        U2_jp2 = load_U(rho2, rhou2, rhov2, E2, i, j+2)

        UL_y1, UR_y1 = reconstruct_state(U2_jm2, U2_jm1, U2_i, U2_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)

        UL_y2, UR_y2 = reconstruct_state(U2_jm1, U2_i, U2_jp1, U2_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)

        local_rhs2 = (Fx_left - Fx_right) / dx + (Fy_bottom - Fy_top) / dy

        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho_new[i, j]  = (1.0 / 3.0) * Un_i[0] + (2.0 / 3.0) * (U2_i[0] + local_rhs2[0] * dt)
        rhou_new[i, j] = (1.0 / 3.0) * Un_i[1] + (2.0 / 3.0) * (U2_i[1] + local_rhs2[1] * dt)
        rhov_new[i, j] = (1.0 / 3.0) * Un_i[2] + (2.0 / 3.0) * (U2_i[2] + local_rhs2[2] * dt)
        E_new[i, j]    = (1.0 / 3.0) * Un_i[3] + (2.0 / 3.0) * (U2_i[3] + local_rhs2[3] * dt)


# ============================================
# UTILITY FUNCTIONS
# ============================================
def get_numpy_state(rho_wp, rhou_wp, rhov_wp, E_wp):
    rho = rho_wp.numpy()
    rhou = rhou_wp.numpy()
    rhov = rhov_wp.numpy()
    E = E_wp.numpy()
    # Stack back to (4, Nx, Ny) for the plotting functions
    return np.stack([rho, rhou, rhov, E], axis=0)

def exact_sod(x, t, gamma=1.4):
    """Exact solution for Sod shock tube problem"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)

    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)

    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)

    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    return f"{base_name}_{scheme}_opti6_SoA_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()

    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)

    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)

    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)

    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])

    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])

    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])

    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])

    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])

    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')

    plt.suptitle(f'2D Riemann Problem Results - {scheme_name} Scheme (RK3), t={t_final:.3f}')
    plt.tight_layout()

    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Final results plot saved as '{filename}'")

    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)

    x = np.linspace(0, 1, NX)

    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) Profiles at t={t_final:.3f}')

    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

    plt.tight_layout()

    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Midplane profiles plot saved as '{midplane_filename}'")

# ============================================
# MAIN GPU SIMULATION LOOP
# ============================================

print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Memory allocation for SoA RK3 stages
rho_n = wp.array(rho_np, dtype=wp.float32, device=device)
rhou_n = wp.array(rhou_np, dtype=wp.float32, device=device)
rhov_n = wp.array(rhov_np, dtype=wp.float32, device=device)
E_n = wp.array(E_np, dtype=wp.float32, device=device)

rho_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

max_speed_arr = wp.zeros(1, dtype=float, device=device)

max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny), inputs=[rho_n, rhou_n, rhov_n, E_n, max_speed_arr, gamma, EPS, NG, Nx, Ny], device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR ) * min(dx, dy) / initial_max_speed

run_times = []

# Argument sets for cleaner launching
args_bc_n = [rho_n, rhou_n, rhov_n, E_n, NG, Nx, Ny]
args_bc_1 = [rho_1, rhou_1, rhov_1, E_1, NG, Nx, Ny]
args_bc_2 = [rho_2, rhou_2, rhov_2, E_2, NG, Nx, Ny]

# --- THE BENCHMARK LOOP ---
for run in range(TOTAL_RUNS):
    # Reset global arrays to Initial Condition for each run
    wp.copy(rho_n, wp.array(rho_np, dtype=wp.float32, device=device))
    wp.copy(rhou_n, wp.array(rhou_np, dtype=wp.float32, device=device))
    wp.copy(rhov_n, wp.array(rhov_np, dtype=wp.float32, device=device))
    wp.copy(E_n, wp.array(E_np, dtype=wp.float32, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    wp.synchronize()
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
              break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=args_bc_n, device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        if TIME_STEPPING == 'euler':
            args_euler = [rho_n, rhou_n, rhov_n, E_n, rho_new, rhou_new, rhov_new, E_new, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny]
            wp.launch(extreme_fused_euler_kernel, dim=(Nx, Ny), block_dim=256, inputs=args_euler, device=device)

            wp.copy(rho_n, rho_new)
            wp.copy(rhou_n, rhou_new)
            wp.copy(rhov_n, rhov_new)
            wp.copy(E_n, E_new)

        elif TIME_STEPPING == 'rk3':
            # RK3 - Stage 1
            args_step1 = [rho_n, rhou_n, rhov_n, E_n, rho_1, rhou_1, rhov_1, E_1, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny]
            wp.launch(extreme_fused_rk3_step1_kernel, dim=(Nx, Ny), block_dim=256, inputs=args_step1, device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=args_bc_1, device=device)

            # RK3 - Stage 2
            args_step2 = [rho_n, rhou_n, rhov_n, E_n, rho_1, rhou_1, rhov_1, E_1, rho_2, rhou_2, rhov_2, E_2, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny]
            wp.launch(extreme_fused_rk3_step2_kernel, dim=(Nx, Ny), block_dim=256, inputs=args_step2, device=device)
            wp.launch(apply_bc_kernel, dim=(Nx, Ny), inputs=args_bc_2, device=device)

            # RK3 - Stage 3
            args_step3 = [rho_n, rhou_n, rhov_n, E_n, rho_2, rhou_2, rhov_2, E_2, rho_new, rhou_new, rhov_new, E_new, gamma, EPS, dx, dy, float(dt), NG, Nx, Ny]
            wp.launch(extreme_fused_rk3_step3_kernel, dim=(Nx, Ny), block_dim=256, inputs=args_step3, device=device)

            # Update next state
            wp.copy(rho_n, rho_new)
            wp.copy(rhou_n, rhou_new)
            wp.copy(rhov_n, rhov_new)
            wp.copy(E_n, E_new)

        t += dt
        step += 1

    wp.synchronize()
    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run + 1}/{TOTAL_RUNS} completed in {elapsed:.4f} seconds")

# ============================================
# AUTOMATED TFLOPS CALCULATION
# ============================================
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding Run 1 (Warm-up): {run_times[0]:.4f}s")
        print(f"Best execution time (Runs 2-{TOTAL_RUNS}): {best_time:.4f} seconds")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} seconds")

    flops_per_cell_per_step = 12000 if TIME_STEPPING == 'rk3' else 4000

    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)

    print(f"Total Steps: {step}")
    print(f"Achieved TFLOPS (Actual Math Workload): {tflops:.6f} TFLOPS")
else:
    print("Profiling complete.")
print("="*50)

# ============================================
# POST-PROCESSING: CREATE FINAL PLOTS
# ============================================

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
print("="*50)

U_final = get_numpy_state(rho_n, rhou_n, rhov_n, E_n)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

# ============================================
# VALIDATION WITH EXACT SOLUTION (CPU)
# ============================================

print("\n" + "="*50)
print("VALIDATION WITH EXACT SOD SOLUTION")
print("="*50)

j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)

x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)

print(f"\nL1 Errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(x, rho_num, 'b-', linewidth=2, label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', linewidth=2, label='Exact Solution')
plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f'Sod Shock Tube Validation - GPU Accelerated, t={t_final}')

plt.subplot(3, 1, 2)
plt.plot(x, u_num, 'b-', linewidth=2)
plt.plot(x, u_ex, 'r--', linewidth=2)
plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(x, p_num, 'b-', linewidth=2)
plt.plot(x, p_ex, 'r--', linewidth=2)
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)

plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"\n✓ Validation plot saved as '{validation_filename}'")
print("\nAll plots generated successfully!")

## opti7

Optimizes thread block dimensions and local stencil loading to prevent register spilling.

In [ ]:
%%writefile FDM_WARP_test_opti7.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
wp.config.verify_fp = False    # Disable NaN/Inf checking
wp.config.fast_math = True     # Enable fast math
wp.init()

if wp.is_cuda_available():
    wp.set_device("cuda:3")     # Adjust to your GPU
    device = "cuda:3"
else:
    device = "cpu"

print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100
TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])

if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE ---\n")
else:
    print(f"\n--- BENCHMARK MODE: {TOTAL_RUNS} runs ---\n")

print(f"Grid: {NX}x{NY}")

NG = 2
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'   # always MUSCL
TIME_STEPPING = 'rk3'        # third‑order SSP
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (SoA NumPy)
# ============================================
rho_np = np.zeros((Nx, Ny), dtype=np.float32)
rhou_np = np.zeros((Nx, Ny), dtype=np.float32)
rhov_np = np.zeros((Nx, Ny), dtype=np.float32)
E_np = np.zeros((Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1
        rho_np[i,j] = rho
        rhou_np[i,j] = rho * u
        rhov_np[i,j] = rho * v
        E_np[i,j] = p/(gamma-1.0) + 0.5 * rho * (u*u + v*v)

# ============================================
# WARP KERNELS (REGISTER‑REDUCED FUSION)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    p = wp.max((gamma - 1.0) * (U[3] - 0.5 * rho * (u*u + v*v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u*u + v*v)
    return wp.vec4f(rho, rho*u, rho*v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    return wp.min(a, b) if a > 0.0 else wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f,
                      gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0,  gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0   - u_m1,   u_p1   - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0   - v_m1,   v_p1   - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0   - p_m1,   p_p1   - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1   - u_0,   u_p2   - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1   - v_0,   v_p2   - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1   - p_0,   p_p2   - p_p1)

    return prim_to_cons(rho_L, u_L, v_L, p_L, gamma), prim_to_cons(rho_R, u_R, v_R, p_R, gamma)

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL*uL + vL*vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR*uR + vR*vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])
    return flux

# -------------------- Helper to load a single cell --------------------
@wp.func
def load_U(rho: wp.array(dtype=wp.float32, ndim=2),
           rhou: wp.array(dtype=wp.float32, ndim=2),
           rhov: wp.array(dtype=wp.float32, ndim=2),
           E: wp.array(dtype=wp.float32, ndim=2),
           i: int, j: int):
    return wp.vec4f(rho[i,j], rhou[i,j], rhov[i,j], E[i,j])

# -------------------- Boundary conditions (SoA) --------------------
@wp.kernel
def apply_bc_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                    rhou: wp.array(dtype=wp.float32, ndim=2),
                    rhov: wp.array(dtype=wp.float32, ndim=2),
                    E: wp.array(dtype=wp.float32, ndim=2),
                    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j
    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1
    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1
    if i != src_i or j != src_j:
        rho[i,j] = rho[src_i, src_j]
        rhou[i,j] = rhou[src_i, src_j]
        rhov[i,j] = rhov[src_i, src_j]
        E[i,j] = E[src_i, src_j]

# -------------------- Compute max speed (for dt) --------------------
@wp.kernel
def compute_max_speed_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                             rhou: wp.array(dtype=wp.float32, ndim=2),
                             rhov: wp.array(dtype=wp.float32, ndim=2),
                             E: wp.array(dtype=wp.float32, ndim=2),
                             max_speed_arr: wp.array(dtype=float),
                             gamma: float, EPS: float,
                             NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U = load_U(rho, rhou, rhov, E, i, j)
        rho_p, u_p, v_p, p_p = cons_to_prim(U, gamma, EPS)
        c = wp.sqrt(gamma * p_p / rho_p)
        speed = wp.abs(u_p) + wp.abs(v_p) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# ---------- EXTREME FUSED RK3 KERNELS (REGISTER‑REDUCED) ----------
# Stage 1: U1 = U + dt * L(U)
@wp.kernel
def rk3_stage1_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2), rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2), E: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Load the 5x5 stencil for this cell (only needed indices)
        U_im2 = load_U(rho, rhou, rhov, E, i-2, j)
        U_im1 = load_U(rho, rhou, rhov, E, i-1, j)
        U_i   = load_U(rho, rhou, rhov, E, i,   j)
        U_ip1 = load_U(rho, rhou, rhov, E, i+1, j)
        U_ip2 = load_U(rho, rhou, rhov, E, i+2, j)
        U_jm2 = load_U(rho, rhou, rhov, E, i,   j-2)
        U_jm1 = load_U(rho, rhou, rhov, E, i,   j-1)
        U_jp1 = load_U(rho, rhou, rhov, E, i,   j+1)
        U_jp2 = load_U(rho, rhou, rhov, E, i,   j+2)

        # ---- X direction fluxes ----
        UL_x1, UR_x1 = reconstruct_state(U_im2, U_im1, U_i, U_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U_im1, U_i, U_ip1, U_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) / dx

        # ---- Y direction fluxes ----
        UL_y1, UR_y1 = reconstruct_state(U_jm2, U_jm1, U_i, U_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U_jm1, U_i, U_jp1, U_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) / dy

        local_rhs = rhs_x + rhs_y
        rho1[i,j]  = U_i[0] + local_rhs[0] * dt
        rhou1[i,j] = U_i[1] + local_rhs[1] * dt
        rhov1[i,j] = U_i[2] + local_rhs[2] * dt
        E1[i,j]    = U_i[3] + local_rhs[3] * dt

# Stage 2: U2 = 0.75*U + 0.25*(U1 + dt*L(U1))
@wp.kernel
def rk3_stage2_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U1_im2 = load_U(rho1, rhou1, rhov1, E1, i-2, j)
        U1_im1 = load_U(rho1, rhou1, rhov1, E1, i-1, j)
        U1_i   = load_U(rho1, rhou1, rhov1, E1, i,   j)
        U1_ip1 = load_U(rho1, rhou1, rhov1, E1, i+1, j)
        U1_ip2 = load_U(rho1, rhou1, rhov1, E1, i+2, j)
        U1_jm2 = load_U(rho1, rhou1, rhov1, E1, i,   j-2)
        U1_jm1 = load_U(rho1, rhou1, rhov1, E1, i,   j-1)
        U1_jp1 = load_U(rho1, rhou1, rhov1, E1, i,   j+1)
        U1_jp2 = load_U(rho1, rhou1, rhov1, E1, i,   j+2)

        # X fluxes on U1
        UL_x1, UR_x1 = reconstruct_state(U1_im2, U1_im1, U1_i, U1_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U1_im1, U1_i, U1_ip1, U1_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) / dx

        # Y fluxes on U1
        UL_y1, UR_y1 = reconstruct_state(U1_jm2, U1_jm1, U1_i, U1_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U1_jm1, U1_i, U1_jp1, U1_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) / dy

        local_rhs1 = rhs_x + rhs_y
        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho2[i,j]  = 0.75 * Un_i[0] + 0.25 * (U1_i[0] + local_rhs1[0] * dt)
        rhou2[i,j] = 0.75 * Un_i[1] + 0.25 * (U1_i[1] + local_rhs1[1] * dt)
        rhov2[i,j] = 0.75 * Un_i[2] + 0.25 * (U1_i[2] + local_rhs1[2] * dt)
        E2[i,j]    = 0.75 * Un_i[3] + 0.25 * (U1_i[3] + local_rhs1[3] * dt)

# Stage 3: U_new = 1/3*U + 2/3*(U2 + dt*L(U2))
@wp.kernel
def rk3_stage3_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    rho_new: wp.array(dtype=wp.float32, ndim=2), rhou_new: wp.array(dtype=wp.float32, ndim=2),
    rhov_new: wp.array(dtype=wp.float32, ndim=2), E_new: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, dx: float, dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U2_im2 = load_U(rho2, rhou2, rhov2, E2, i-2, j)
        U2_im1 = load_U(rho2, rhou2, rhov2, E2, i-1, j)
        U2_i   = load_U(rho2, rhou2, rhov2, E2, i,   j)
        U2_ip1 = load_U(rho2, rhou2, rhov2, E2, i+1, j)
        U2_ip2 = load_U(rho2, rhou2, rhov2, E2, i+2, j)
        U2_jm2 = load_U(rho2, rhou2, rhov2, E2, i,   j-2)
        U2_jm1 = load_U(rho2, rhou2, rhov2, E2, i,   j-1)
        U2_jp1 = load_U(rho2, rhou2, rhov2, E2, i,   j+1)
        U2_jp2 = load_U(rho2, rhou2, rhov2, E2, i,   j+2)

        # X fluxes on U2
        UL_x1, UR_x1 = reconstruct_state(U2_im2, U2_im1, U2_i, U2_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U2_im1, U2_i, U2_ip1, U2_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) / dx

        # Y fluxes on U2
        UL_y1, UR_y1 = reconstruct_state(U2_jm2, U2_jm1, U2_i, U2_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U2_jm1, U2_i, U2_jp1, U2_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) / dy

        local_rhs2 = rhs_x + rhs_y
        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho_new[i,j]  = (1.0/3.0) * Un_i[0] + (2.0/3.0) * (U2_i[0] + local_rhs2[0] * dt)
        rhou_new[i,j] = (1.0/3.0) * Un_i[1] + (2.0/3.0) * (U2_i[1] + local_rhs2[1] * dt)
        rhov_new[i,j] = (1.0/3.0) * Un_i[2] + (2.0/3.0) * (U2_i[2] + local_rhs2[2] * dt)
        E_new[i,j]    = (1.0/3.0) * Un_i[3] + (2.0/3.0) * (U2_i[3] + local_rhs2[3] * dt)

# ============================================
# UTILITY FUNCTIONS (unchanged from opti6)
# ============================================
def get_numpy_state(rho_wp, rhou_wp, rhov_wp, E_wp):
    return np.stack([rho_wp.numpy(), rhou_wp.numpy(), rhov_wp.numpy(), E_wp.numpy()], axis=0)

def exact_sod(x, t, gamma=1.4):
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1
    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)
    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)
    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)
    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL:
            rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star:
            rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR:
            rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else:
            rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    return f"{base_name}_{scheme}_opti7_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])
    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')
    plt.suptitle(f'2D Riemann Problem - {scheme_name} (opti7), t={t_final:.3f}')
    plt.tight_layout()
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ {filename}")
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)
    x = np.linspace(0, 1, NX)
    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) at t={t_final:.3f}')
    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)
    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ {midplane_filename}")

# ============================================
# MAIN SIMULATION LOOP
# ============================================
print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate SoA arrays
rho_n = wp.array(rho_np, dtype=wp.float32, device=device)
rhou_n = wp.array(rhou_np, dtype=wp.float32, device=device)
rhov_n = wp.array(rhov_np, dtype=wp.float32, device=device)
E_n = wp.array(E_np, dtype=wp.float32, device=device)

rho_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

max_speed_arr = wp.zeros(1, dtype=float, device=device)

# Compute fixed dt
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny),
          inputs=[rho_n, rhou_n, rhov_n, E_n, max_speed_arr, gamma, EPS, NG, Nx, Ny],
          device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR) * min(dx, dy) / initial_max_speed
print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps: {int(t_final/dt_fixed) + 1}")

run_times = []

# Benchmark loop
for run in range(TOTAL_RUNS):
    wp.copy(rho_n, wp.array(rho_np, dtype=wp.float32, device=device))
    wp.copy(rhou_n, wp.array(rhou_np, dtype=wp.float32, device=device))
    wp.copy(rhov_n, wp.array(rhov_np, dtype=wp.float32, device=device))
    wp.copy(E_n, wp.array(E_np, dtype=wp.float32, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
            break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        # RK3 stages with reduced register usage and block_dim=128
        wp.launch(rk3_stage1_kernel, dim=(Nx, Ny), block_dim=128,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_1, rhou_1, rhov_1, E_1, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage2_kernel, dim=(Nx, Ny), block_dim=128,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          rho_2, rhou_2, rhov_2, E_2,
                          gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_2, rhou_2, rhov_2, E_2, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage3_kernel, dim=(Nx, Ny), block_dim=128,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_2, rhou_2, rhov_2, E_2,
                          rho_new, rhou_new, rhov_new, E_new,
                          gamma, EPS, dx, dy, float(dt), NG, Nx, Ny], device=device)

        # Update state
        wp.copy(rho_n, rho_new)
        wp.copy(rhou_n, rhou_new)
        wp.copy(rhov_n, rhov_new)
        wp.copy(E_n, E_new)

        t += dt
        step += 1

    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run+1}/{TOTAL_RUNS} completed in {elapsed:.4f} s")

# TFLOPS reporting (using the same flop count as opti6 for comparison)
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding warm-up: {run_times[0]:.4f} s")
        print(f"Best time (runs 2-{TOTAL_RUNS}): {best_time:.4f} s")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} s")

    flops_per_cell_per_step = 12000   # same as opti6 (RK3)
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)
    print(f"Total steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling mode – ignoring timing.")
print("="*50)

# Post‑processing: plots and validation
print("\n" + "="*50)
print("CREATING FINAL PLOTS")
U_final = get_numpy_state(rho_n, rhou_n, rhov_n, E_n)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

print("\nVALIDATION WITH EXACT SOD SOLUTION")
j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)
x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)
print(f"L1 errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12,10))
plt.subplot(3,1,1)
plt.plot(x, rho_num, 'b-', label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', label='Exact')
plt.ylabel('Density'); plt.legend(); plt.grid(True)
plt.title(f'Sod shock tube validation, t={t_final}')
plt.subplot(3,1,2)
plt.plot(x, u_num, 'b-'); plt.plot(x, u_ex, 'r--')
plt.ylabel('Velocity'); plt.grid(True)
plt.subplot(3,1,3)
plt.plot(x, p_num, 'b-'); plt.plot(x, p_ex, 'r--')
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True)
plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"✓ {validation_filename}\nAll plots generated.")


## opti8

Pushes grid spacing calculations to constants to squeeze out final floating-point efficiencies.

In [ ]:
%%writefile FDM_WARP_test_opti8.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
wp.config.verify_fp = False    # Disable NaN/Inf checking
wp.config.fast_math = True     # Enable fast math
wp.init()

if wp.is_cuda_available():
    wp.set_device("cuda:2")     # Adjust to your GPU (0,1,2,3)
    device = "cuda:2"
else:
    device = "cpu"

print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100

TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])
if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE ---\n")
else:
    print(f"\n--- BENCHMARK MODE: {TOTAL_RUNS} runs ---\n")

print(f"Grid: {NX}×{NY}")

NG = 2                      # Ghost cells
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY
inv_dx = 1.0 / dx           # Precomputed inverse for speed
inv_dy = 1.0 / dy

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'   # Always MUSCL
TIME_STEPPING = 'rk3'        # SSP-RK3
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (SoA NumPy)
# ============================================
rho_np = np.zeros((Nx, Ny), dtype=np.float32)
rhou_np = np.zeros((Nx, Ny), dtype=np.float32)
rhov_np = np.zeros((Nx, Ny), dtype=np.float32)
E_np = np.zeros((Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1
        rho_np[i,j] = rho
        rhou_np[i,j] = rho * u
        rhov_np[i,j] = rho * v
        E_np[i,j] = p/(gamma-1.0) + 0.5 * rho * (u*u + v*v)

# ============================================
# WARP KERNELS (EXTREME FUSION, SoA, BLOCK_DIM=256)
# ============================================

@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u*u + v*v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u*u + v*v)
    return wp.vec4f(rho, rho*u, rho*v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    return wp.min(a, b) if a > 0.0 else wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f,
                      gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0,  gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    # Left state
    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0   - u_m1,   u_p1   - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0   - v_m1,   v_p1   - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0   - p_m1,   p_p1   - p_0)

    # Right state
    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1   - u_0,   u_p2   - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1   - v_0,   v_p2   - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1   - p_0,   p_p2   - p_p1)

    return prim_to_cons(rho_L, u_L, v_L, p_L, gamma), prim_to_cons(rho_R, u_R, v_R, p_R, gamma)

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL*uL + vL*vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR*uR + vR*vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])
    return flux

@wp.func
def load_U(rho: wp.array(dtype=wp.float32, ndim=2),
           rhou: wp.array(dtype=wp.float32, ndim=2),
           rhov: wp.array(dtype=wp.float32, ndim=2),
           E: wp.array(dtype=wp.float32, ndim=2),
           i: int, j: int):
    return wp.vec4f(rho[i,j], rhou[i,j], rhov[i,j], E[i,j])

@wp.kernel
def apply_bc_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                    rhou: wp.array(dtype=wp.float32, ndim=2),
                    rhov: wp.array(dtype=wp.float32, ndim=2),
                    E: wp.array(dtype=wp.float32, ndim=2),
                    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j
    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1
    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1
    if i != src_i or j != src_j:
        rho[i,j] = rho[src_i, src_j]
        rhou[i,j] = rhou[src_i, src_j]
        rhov[i,j] = rhov[src_i, src_j]
        E[i,j] = E[src_i, src_j]

@wp.kernel
def compute_max_speed_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                             rhou: wp.array(dtype=wp.float32, ndim=2),
                             rhov: wp.array(dtype=wp.float32, ndim=2),
                             E: wp.array(dtype=wp.float32, ndim=2),
                             max_speed_arr: wp.array(dtype=float),
                             gamma: float, EPS: float,
                             NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U = load_U(rho, rhou, rhov, E, i, j)
        rho_p, u_p, v_p, p_p = cons_to_prim(U, gamma, EPS)
        c = wp.sqrt(gamma * p_p / rho_p)
        speed = wp.abs(u_p) + wp.abs(v_p) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# ---------- RK3 Stage 1: U1 = U + dt * L(U) ----------
@wp.kernel
def rk3_stage1_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2), rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2), E: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Load stencil
        U_im2 = load_U(rho, rhou, rhov, E, i-2, j)
        U_im1 = load_U(rho, rhou, rhov, E, i-1, j)
        U_i   = load_U(rho, rhou, rhov, E, i,   j)
        U_ip1 = load_U(rho, rhou, rhov, E, i+1, j)
        U_ip2 = load_U(rho, rhou, rhov, E, i+2, j)
        U_jm2 = load_U(rho, rhou, rhov, E, i,   j-2)
        U_jm1 = load_U(rho, rhou, rhov, E, i,   j-1)
        U_jp1 = load_U(rho, rhou, rhov, E, i,   j+1)
        U_jp2 = load_U(rho, rhou, rhov, E, i,   j+2)

        # X-direction fluxes
        UL_x1, UR_x1 = reconstruct_state(U_im2, U_im1, U_i, U_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U_im1, U_i, U_ip1, U_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y-direction fluxes
        UL_y1, UR_y1 = reconstruct_state(U_jm2, U_jm1, U_i, U_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U_jm1, U_i, U_jp1, U_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        rho1[i,j]  = U_i[0] + local_rhs[0] * dt
        rhou1[i,j] = U_i[1] + local_rhs[1] * dt
        rhov1[i,j] = U_i[2] + local_rhs[2] * dt
        E1[i,j]    = U_i[3] + local_rhs[3] * dt

# ---------- RK3 Stage 2: U2 = 0.75*U + 0.25*(U1 + dt*L(U1)) ----------
@wp.kernel
def rk3_stage2_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Load stencil from U1
        U1_im2 = load_U(rho1, rhou1, rhov1, E1, i-2, j)
        U1_im1 = load_U(rho1, rhou1, rhov1, E1, i-1, j)
        U1_i   = load_U(rho1, rhou1, rhov1, E1, i,   j)
        U1_ip1 = load_U(rho1, rhou1, rhov1, E1, i+1, j)
        U1_ip2 = load_U(rho1, rhou1, rhov1, E1, i+2, j)
        U1_jm2 = load_U(rho1, rhou1, rhov1, E1, i,   j-2)
        U1_jm1 = load_U(rho1, rhou1, rhov1, E1, i,   j-1)
        U1_jp1 = load_U(rho1, rhou1, rhov1, E1, i,   j+1)
        U1_jp2 = load_U(rho1, rhou1, rhov1, E1, i,   j+2)

        # X fluxes on U1
        UL_x1, UR_x1 = reconstruct_state(U1_im2, U1_im1, U1_i, U1_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U1_im1, U1_i, U1_ip1, U1_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y fluxes on U1
        UL_y1, UR_y1 = reconstruct_state(U1_jm2, U1_jm1, U1_i, U1_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U1_jm1, U1_i, U1_jp1, U1_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho2[i,j]  = 0.75 * Un_i[0] + 0.25 * (U1_i[0] + local_rhs[0] * dt)
        rhou2[i,j] = 0.75 * Un_i[1] + 0.25 * (U1_i[1] + local_rhs[1] * dt)
        rhov2[i,j] = 0.75 * Un_i[2] + 0.25 * (U1_i[2] + local_rhs[2] * dt)
        E2[i,j]    = 0.75 * Un_i[3] + 0.25 * (U1_i[3] + local_rhs[3] * dt)

# ---------- RK3 Stage 3: U_new = 1/3*U + 2/3*(U2 + dt*L(U2)) ----------
@wp.kernel
def rk3_stage3_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    rho_new: wp.array(dtype=wp.float32, ndim=2), rhou_new: wp.array(dtype=wp.float32, ndim=2),
    rhov_new: wp.array(dtype=wp.float32, ndim=2), E_new: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        # Load stencil from U2
        U2_im2 = load_U(rho2, rhou2, rhov2, E2, i-2, j)
        U2_im1 = load_U(rho2, rhou2, rhov2, E2, i-1, j)
        U2_i   = load_U(rho2, rhou2, rhov2, E2, i,   j)
        U2_ip1 = load_U(rho2, rhou2, rhov2, E2, i+1, j)
        U2_ip2 = load_U(rho2, rhou2, rhov2, E2, i+2, j)
        U2_jm2 = load_U(rho2, rhou2, rhov2, E2, i,   j-2)
        U2_jm1 = load_U(rho2, rhou2, rhov2, E2, i,   j-1)
        U2_jp1 = load_U(rho2, rhou2, rhov2, E2, i,   j+1)
        U2_jp2 = load_U(rho2, rhou2, rhov2, E2, i,   j+2)

        # X fluxes on U2
        UL_x1, UR_x1 = reconstruct_state(U2_im2, U2_im1, U2_i, U2_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U2_im1, U2_i, U2_ip1, U2_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y fluxes on U2
        UL_y1, UR_y1 = reconstruct_state(U2_jm2, U2_jm1, U2_i, U2_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U2_jm1, U2_i, U2_jp1, U2_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        Un_i = load_U(rho_n, rhou_n, rhov_n, E_n, i, j)

        rho_new[i,j]  = (1.0/3.0) * Un_i[0] + (2.0/3.0) * (U2_i[0] + local_rhs[0] * dt)
        rhou_new[i,j] = (1.0/3.0) * Un_i[1] + (2.0/3.0) * (U2_i[1] + local_rhs[1] * dt)
        rhov_new[i,j] = (1.0/3.0) * Un_i[2] + (2.0/3.0) * (U2_i[2] + local_rhs[2] * dt)
        E_new[i,j]    = (1.0/3.0) * Un_i[3] + (2.0/3.0) * (U2_i[3] + local_rhs[3] * dt)

# ============================================
# UTILITY FUNCTIONS (unchanged)
# ============================================
def get_numpy_state(rho_wp, rhou_wp, rhov_wp, E_wp):
    return np.stack([rho_wp.numpy(), rhou_wp.numpy(), rhov_wp.numpy(), E_wp.numpy()], axis=0)

def exact_sod(x, t, gamma=1.4):
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1
    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)
    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)
    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)
    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    return f"{base_name}_{scheme}_opti8_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])
    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')
    plt.suptitle(f'2D Riemann Problem - {scheme_name} (opti8), t={t_final:.3f}')
    plt.tight_layout()
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ {filename}")
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)
    x = np.linspace(0, 1, NX)
    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) at t={t_final:.3f}')
    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)
    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ {midplane_filename}")

# ============================================
# MAIN SIMULATION LOOP
# ============================================
print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate SoA arrays
rho_n = wp.array(rho_np, dtype=wp.float32, device=device)
rhou_n = wp.array(rhou_np, dtype=wp.float32, device=device)
rhov_n = wp.array(rhov_np, dtype=wp.float32, device=device)
E_n = wp.array(E_np, dtype=wp.float32, device=device)

rho_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

max_speed_arr = wp.zeros(1, dtype=float, device=device)

# Compute fixed dt
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny),
          inputs=[rho_n, rhou_n, rhov_n, E_n, max_speed_arr, gamma, EPS, NG, Nx, Ny],
          device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR) * min(dx, dy) / initial_max_speed
print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps: {int(t_final/dt_fixed) + 1}")

run_times = []

# Benchmark loop
for run in range(TOTAL_RUNS):
    wp.copy(rho_n, wp.array(rho_np, dtype=wp.float32, device=device))
    wp.copy(rhou_n, wp.array(rhou_np, dtype=wp.float32, device=device))
    wp.copy(rhov_n, wp.array(rhov_np, dtype=wp.float32, device=device))
    wp.copy(E_n, wp.array(E_np, dtype=wp.float32, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
            break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        # RK3 stages with block_dim=256
        wp.launch(rk3_stage1_kernel, dim=(Nx, Ny), block_dim=256,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_1, rhou_1, rhov_1, E_1, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage2_kernel, dim=(Nx, Ny), block_dim=256,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          rho_2, rhou_2, rhov_2, E_2,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_2, rhou_2, rhov_2, E_2, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage3_kernel, dim=(Nx, Ny), block_dim=256,
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_2, rhou_2, rhov_2, E_2,
                          rho_new, rhou_new, rhov_new, E_new,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)

        # Update state
        wp.copy(rho_n, rho_new)
        wp.copy(rhou_n, rhou_new)
        wp.copy(rhov_n, rhov_new)
        wp.copy(E_n, E_new)

        t += dt
        step += 1

    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run+1}/{TOTAL_RUNS} completed in {elapsed:.4f} s")

# TFLOPS reporting
print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding warm-up: {run_times[0]:.4f} s")
        print(f"Best time (runs 2-{TOTAL_RUNS}): {best_time:.4f} s")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} s")

    flops_per_cell_per_step = 12000   # RK3
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)
    print(f"Total steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling mode – ignoring timing.")
print("="*50)

# Post‑processing: plots and validation
print("\n" + "="*50)
print("CREATING FINAL PLOTS")
U_final = get_numpy_state(rho_n, rhou_n, rhov_n, E_n)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

print("\nVALIDATION WITH EXACT SOD SOLUTION")
j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)
x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)
print(f"L1 errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12,10))
plt.subplot(3,1,1)
plt.plot(x, rho_num, 'b-', label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', label='Exact')
plt.ylabel('Density'); plt.legend(); plt.grid(True)
plt.title(f'Sod shock tube validation, t={t_final}')
plt.subplot(3,1,2)
plt.plot(x, u_num, 'b-'); plt.plot(x, u_ex, 'r--')
plt.ylabel('Velocity'); plt.grid(True)
plt.subplot(3,1,3)
plt.plot(x, p_num, 'b-'); plt.plot(x, p_ex, 'r--')
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True)
plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"✓ {validation_filename}\nAll plots generated.")

## opti8_shmem

Leverages ultra-fast on-chip GPU shared memory to cache overlapping cell data.

In [ ]:
%%writefile FDM_WARP_test_opti8_shmem.py
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import warp as wp

# ============================================
# WARP INITIALIZATION & OPTIMIZATIONS
# ============================================
wp.config.verify_fp = False
wp.config.fast_math = True
wp.init()

if wp.is_cuda_available():
    wp.set_device("cuda:2")
    device = "cuda:2"
else:
    device = "cpu"

print("Warp devices:", wp.get_devices())
print("Current device:", wp.get_device())

# ============================================
# PARAMETERS
# ============================================
gamma = 1.4
NX, NY = 500, 100
TOTAL_RUNS = 7
IS_PROFILING = False

if len(sys.argv) >= 3:
    NX = int(sys.argv[1])
    NY = int(sys.argv[2])
if len(sys.argv) == 4 and sys.argv[3] == '--profile':
    TOTAL_RUNS = 1
    IS_PROFILING = True
    print("\n--- PROFILING MODE ACTIVE ---\n")
else:
    print(f"\n--- BENCHMARK MODE: {TOTAL_RUNS} runs ---\n")

print(f"Grid: {NX}×{NY}")

NG = 2
Nx, Ny = NX + 2*NG, NY + 2*NG
dx = 1.0 / NX
dy = 1.0 / NY
inv_dx = 1.0 / dx
inv_dy = 1.0 / dy

CFL = 0.5
t_final = 0.2
EPS = 1e-8

NUMERICAL_SCHEME = 'muscl'
TIME_STEPPING = 'rk3'
PRINT_EVERY_N_STEPS = 20

# ============================================
# INITIAL CONDITION (SoA NumPy)
# ============================================
rho_np = np.zeros((Nx, Ny), dtype=np.float32)
rhou_np = np.zeros((Nx, Ny), dtype=np.float32)
rhov_np = np.zeros((Nx, Ny), dtype=np.float32)
E_np = np.zeros((Nx, Ny), dtype=np.float32)

for i in range(Nx):
    for j in range(Ny):
        if i < Nx//2:
            rho, u, v, p = 1.0, 0.0, 0.0, 1.0
        else:
            rho, u, v, p = 0.125, 0.0, 0.0, 0.1
        rho_np[i,j] = rho
        rhou_np[i,j] = rho * u
        rhov_np[i,j] = rho * v
        E_np[i,j] = p/(gamma-1.0) + 0.5 * rho * (u*u + v*v)

# ============================================
# WARP KERNELS
# ============================================
@wp.func
def cons_to_prim(U: wp.vec4f, gamma: float, EPS: float):
    rho = wp.max(U[0], EPS)
    inv_rho = 1.0 / rho
    u = U[1] * inv_rho
    v = U[2] * inv_rho
    E = U[3]
    p = wp.max((gamma - 1.0) * (E - 0.5 * rho * (u*u + v*v)), EPS)
    return rho, u, v, p

@wp.func
def prim_to_cons(rho: float, u: float, v: float, p: float, gamma: float):
    E = p / (gamma - 1.0) + 0.5 * rho * (u*u + v*v)
    return wp.vec4f(rho, rho*u, rho*v, E)

@wp.func
def minmod(a: float, b: float):
    if a * b <= 0.0:
        return 0.0
    return wp.min(a, b) if a > 0.0 else wp.max(a, b)

@wp.func
def reconstruct_state(U_m1: wp.vec4f, U_0: wp.vec4f, U_p1: wp.vec4f, U_p2: wp.vec4f,
                      gamma: float, EPS: float):
    rho_m1, u_m1, v_m1, p_m1 = cons_to_prim(U_m1, gamma, EPS)
    rho_0,  u_0,  v_0,  p_0  = cons_to_prim(U_0,  gamma, EPS)
    rho_p1, u_p1, v_p1, p_p1 = cons_to_prim(U_p1, gamma, EPS)
    rho_p2, u_p2, v_p2, p_p2 = cons_to_prim(U_p2, gamma, EPS)

    rho_L = rho_0 + 0.5 * minmod(rho_0 - rho_m1, rho_p1 - rho_0)
    u_L   = u_0   + 0.5 * minmod(u_0   - u_m1,   u_p1   - u_0)
    v_L   = v_0   + 0.5 * minmod(v_0   - v_m1,   v_p1   - v_0)
    p_L   = p_0   + 0.5 * minmod(p_0   - p_m1,   p_p1   - p_0)

    rho_R = rho_p1 - 0.5 * minmod(rho_p1 - rho_0, rho_p2 - rho_p1)
    u_R   = u_p1   - 0.5 * minmod(u_p1   - u_0,   u_p2   - u_p1)
    v_R   = v_p1   - 0.5 * minmod(v_p1   - v_0,   v_p2   - v_p1)
    p_R   = p_p1   - 0.5 * minmod(p_p1   - p_0,   p_p2   - p_p1)

    return prim_to_cons(rho_L, u_L, v_L, p_L, gamma), prim_to_cons(rho_R, u_R, v_R, p_R, gamma)

@wp.func
def hllc_flux(UL: wp.vec4f, UR: wp.vec4f, gamma: float, EPS: float, dir_y: int):
    if dir_y == 1:
        UL = wp.vec4f(UL[0], UL[2], UL[1], UL[3])
        UR = wp.vec4f(UR[0], UR[2], UR[1], UR[3])

    rhoL = wp.max(UL[0], EPS)
    inv_rhoL = 1.0 / rhoL
    uL = UL[1] * inv_rhoL
    vL = UL[2] * inv_rhoL
    pL = wp.max((gamma - 1.0) * (UL[3] - 0.5 * rhoL * (uL*uL + vL*vL)), EPS)

    rhoR = wp.max(UR[0], EPS)
    inv_rhoR = 1.0 / rhoR
    uR = UR[1] * inv_rhoR
    vR = UR[2] * inv_rhoR
    pR = wp.max((gamma - 1.0) * (UR[3] - 0.5 * rhoR * (uR*uR + vR*vR)), EPS)

    cL = wp.sqrt(gamma * pL * inv_rhoL)
    cR = wp.sqrt(gamma * pR * inv_rhoR)

    sqrt_rhoL = wp.sqrt(rhoL)
    sqrt_rhoR = wp.sqrt(rhoR)
    u_avg = (sqrt_rhoL * uL + sqrt_rhoR * uR) / (sqrt_rhoL + sqrt_rhoR)

    inv_sum_rho = 1.0 / (rhoL + rhoR)
    c_avg = wp.sqrt(gamma * (pL + pR) * inv_sum_rho)

    SL = wp.min(uL - cL, u_avg - c_avg)
    SR = wp.max(uR + cR, u_avg + c_avg)

    FL = wp.vec4f(rhoL * uL, rhoL * uL * uL + pL, rhoL * uL * vL, uL * (UL[3] + pL))
    FR = wp.vec4f(rhoR * uR, rhoR * uR * uR + pR, rhoR * uR * vR, uR * (UR[3] + pR))

    if SL >= 0.0:
        flux = FL
    elif SR <= 0.0:
        flux = FR
    else:
        inv_SR_SL = 1.0 / (SR - SL)
        flux = (SR * FL - SL * FR + SL * SR * (UR - UL)) * inv_SR_SL

    if dir_y == 1:
        flux = wp.vec4f(flux[0], flux[2], flux[1], flux[3])
    return flux

# ------------------------------------------------------------
# Boundary conditions
# ------------------------------------------------------------
@wp.kernel
def apply_bc_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                    rhou: wp.array(dtype=wp.float32, ndim=2),
                    rhov: wp.array(dtype=wp.float32, ndim=2),
                    E: wp.array(dtype=wp.float32, ndim=2),
                    NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    src_i, src_j = i, j
    if i < NG: src_i = NG
    elif i >= Nx - NG: src_i = Nx - NG - 1
    if j < NG: src_j = NG
    elif j >= Ny - NG: src_j = Ny - NG - 1
    if i != src_i or j != src_j:
        rho[i,j] = rho[src_i, src_j]
        rhou[i,j] = rhou[src_i, src_j]
        rhov[i,j] = rhov[src_i, src_j]
        E[i,j] = E[src_i, src_j]

# ------------------------------------------------------------
# Compute max speed (for dt)
# ------------------------------------------------------------
@wp.kernel
def compute_max_speed_kernel(rho: wp.array(dtype=wp.float32, ndim=2),
                             rhou: wp.array(dtype=wp.float32, ndim=2),
                             rhov: wp.array(dtype=wp.float32, ndim=2),
                             E: wp.array(dtype=wp.float32, ndim=2),
                             max_speed_arr: wp.array(dtype=float),
                             gamma: float, EPS: float,
                             NG: int, Nx: int, Ny: int):
    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U = wp.vec4f(rho[i,j], rhou[i,j], rhov[i,j], E[i,j])
        rho_p, u_p, v_p, p_p = cons_to_prim(U, gamma, EPS)
        c = wp.sqrt(gamma * p_p / rho_p)
        speed = wp.abs(u_p) + wp.abs(v_p) + c
        wp.atomic_max(max_speed_arr, 0, speed)

# ------------------------------------------------------------
# RK3 Stage 1 (Global Memory)
# ------------------------------------------------------------
@wp.kernel
def rk3_stage1_kernel(
    rho: wp.array(dtype=wp.float32, ndim=2), rhou: wp.array(dtype=wp.float32, ndim=2),
    rhov: wp.array(dtype=wp.float32, ndim=2), E: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):

    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U_im2 = wp.vec4f(rho[i-2, j], rhou[i-2, j], rhov[i-2, j], E[i-2, j])
        U_im1 = wp.vec4f(rho[i-1, j], rhou[i-1, j], rhov[i-1, j], E[i-1, j])
        U_i   = wp.vec4f(rho[i,   j], rhou[i,   j], rhov[i,   j], E[i,   j])
        U_ip1 = wp.vec4f(rho[i+1, j], rhou[i+1, j], rhov[i+1, j], E[i+1, j])
        U_ip2 = wp.vec4f(rho[i+2, j], rhou[i+2, j], rhov[i+2, j], E[i+2, j])

        U_jm2 = wp.vec4f(rho[i, j-2], rhou[i, j-2], rhov[i, j-2], E[i, j-2])
        U_jm1 = wp.vec4f(rho[i, j-1], rhou[i, j-1], rhov[i, j-1], E[i, j-1])
        U_jp1 = wp.vec4f(rho[i, j+1], rhou[i, j+1], rhov[i, j+1], E[i, j+1])
        U_jp2 = wp.vec4f(rho[i, j+2], rhou[i, j+2], rhov[i, j+2], E[i, j+2])

        # X fluxes
        UL_x1, UR_x1 = reconstruct_state(U_im2, U_im1, U_i, U_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U_im1, U_i, U_ip1, U_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y fluxes
        UL_y1, UR_y1 = reconstruct_state(U_jm2, U_jm1, U_i, U_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U_jm1, U_i, U_jp1, U_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        rho1[i,j]  = U_i[0] + local_rhs[0] * dt
        rhou1[i,j] = U_i[1] + local_rhs[1] * dt
        rhov1[i,j] = U_i[2] + local_rhs[2] * dt
        E1[i,j]    = U_i[3] + local_rhs[3] * dt

# ------------------------------------------------------------
# RK3 Stage 2 (Global Memory)
# ------------------------------------------------------------
@wp.kernel
def rk3_stage2_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho1: wp.array(dtype=wp.float32, ndim=2), rhou1: wp.array(dtype=wp.float32, ndim=2),
    rhov1: wp.array(dtype=wp.float32, ndim=2), E1: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):

    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U1_im2 = wp.vec4f(rho1[i-2, j], rhou1[i-2, j], rhov1[i-2, j], E1[i-2, j])
        U1_im1 = wp.vec4f(rho1[i-1, j], rhou1[i-1, j], rhov1[i-1, j], E1[i-1, j])
        U1_i   = wp.vec4f(rho1[i,   j], rhou1[i,   j], rhov1[i,   j], E1[i,   j])
        U1_ip1 = wp.vec4f(rho1[i+1, j], rhou1[i+1, j], rhov1[i+1, j], E1[i+1, j])
        U1_ip2 = wp.vec4f(rho1[i+2, j], rhou1[i+2, j], rhov1[i+2, j], E1[i+2, j])

        U1_jm2 = wp.vec4f(rho1[i, j-2], rhou1[i, j-2], rhov1[i, j-2], E1[i, j-2])
        U1_jm1 = wp.vec4f(rho1[i, j-1], rhou1[i, j-1], rhov1[i, j-1], E1[i, j-1])
        U1_jp1 = wp.vec4f(rho1[i, j+1], rhou1[i, j+1], rhov1[i, j+1], E1[i, j+1])
        U1_jp2 = wp.vec4f(rho1[i, j+2], rhou1[i, j+2], rhov1[i, j+2], E1[i, j+2])

        # X fluxes
        UL_x1, UR_x1 = reconstruct_state(U1_im2, U1_im1, U1_i, U1_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U1_im1, U1_i, U1_ip1, U1_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y fluxes
        UL_y1, UR_y1 = reconstruct_state(U1_jm2, U1_jm1, U1_i, U1_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U1_jm1, U1_i, U1_jp1, U1_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        Un_i = wp.vec4f(rho_n[i,j], rhou_n[i,j], rhov_n[i,j], E_n[i,j])

        rho2[i,j]  = 0.75 * Un_i[0] + 0.25 * (U1_i[0] + local_rhs[0] * dt)
        rhou2[i,j] = 0.75 * Un_i[1] + 0.25 * (U1_i[1] + local_rhs[1] * dt)
        rhov2[i,j] = 0.75 * Un_i[2] + 0.25 * (U1_i[2] + local_rhs[2] * dt)
        E2[i,j]    = 0.75 * Un_i[3] + 0.25 * (U1_i[3] + local_rhs[3] * dt)

# ------------------------------------------------------------
# RK3 Stage 3 (Global Memory)
# ------------------------------------------------------------
@wp.kernel
def rk3_stage3_kernel(
    rho_n: wp.array(dtype=wp.float32, ndim=2), rhou_n: wp.array(dtype=wp.float32, ndim=2),
    rhov_n: wp.array(dtype=wp.float32, ndim=2), E_n: wp.array(dtype=wp.float32, ndim=2),
    rho2: wp.array(dtype=wp.float32, ndim=2), rhou2: wp.array(dtype=wp.float32, ndim=2),
    rhov2: wp.array(dtype=wp.float32, ndim=2), E2: wp.array(dtype=wp.float32, ndim=2),
    rho_new: wp.array(dtype=wp.float32, ndim=2), rhou_new: wp.array(dtype=wp.float32, ndim=2),
    rhov_new: wp.array(dtype=wp.float32, ndim=2), E_new: wp.array(dtype=wp.float32, ndim=2),
    gamma: float, EPS: float, inv_dx: float, inv_dy: float, dt: float,
    NG: int, Nx: int, Ny: int):

    i, j = wp.tid()
    if i >= NG and i < Nx - NG and j >= NG and j < Ny - NG:
        U2_im2 = wp.vec4f(rho2[i-2, j], rhou2[i-2, j], rhov2[i-2, j], E2[i-2, j])
        U2_im1 = wp.vec4f(rho2[i-1, j], rhou2[i-1, j], rhov2[i-1, j], E2[i-1, j])
        U2_i   = wp.vec4f(rho2[i,   j], rhou2[i,   j], rhov2[i,   j], E2[i,   j])
        U2_ip1 = wp.vec4f(rho2[i+1, j], rhou2[i+1, j], rhov2[i+1, j], E2[i+1, j])
        U2_ip2 = wp.vec4f(rho2[i+2, j], rhou2[i+2, j], rhov2[i+2, j], E2[i+2, j])

        U2_jm2 = wp.vec4f(rho2[i, j-2], rhou2[i, j-2], rhov2[i, j-2], E2[i, j-2])
        U2_jm1 = wp.vec4f(rho2[i, j-1], rhou2[i, j-1], rhov2[i, j-1], E2[i, j-1])
        U2_jp1 = wp.vec4f(rho2[i, j+1], rhou2[i, j+1], rhov2[i, j+1], E2[i, j+1])
        U2_jp2 = wp.vec4f(rho2[i, j+2], rhou2[i, j+2], rhov2[i, j+2], E2[i, j+2])

        # X fluxes
        UL_x1, UR_x1 = reconstruct_state(U2_im2, U2_im1, U2_i, U2_ip1, gamma, EPS)
        Fx_left = hllc_flux(UL_x1, UR_x1, gamma, EPS, 0)
        UL_x2, UR_x2 = reconstruct_state(U2_im1, U2_i, U2_ip1, U2_ip2, gamma, EPS)
        Fx_right = hllc_flux(UL_x2, UR_x2, gamma, EPS, 0)
        rhs_x = (Fx_left - Fx_right) * inv_dx

        # Y fluxes
        UL_y1, UR_y1 = reconstruct_state(U2_jm2, U2_jm1, U2_i, U2_jp1, gamma, EPS)
        Fy_bottom = hllc_flux(UL_y1, UR_y1, gamma, EPS, 1)
        UL_y2, UR_y2 = reconstruct_state(U2_jm1, U2_i, U2_jp1, U2_jp2, gamma, EPS)
        Fy_top = hllc_flux(UL_y2, UR_y2, gamma, EPS, 1)
        rhs_y = (Fy_bottom - Fy_top) * inv_dy

        local_rhs = rhs_x + rhs_y
        Un_i = wp.vec4f(rho_n[i,j], rhou_n[i,j], rhov_n[i,j], E_n[i,j])

        rho_new[i,j]  = (1.0/3.0) * Un_i[0] + (2.0/3.0) * (U2_i[0] + local_rhs[0] * dt)
        rhou_new[i,j] = (1.0/3.0) * Un_i[1] + (2.0/3.0) * (U2_i[1] + local_rhs[1] * dt)
        rhov_new[i,j] = (1.0/3.0) * Un_i[2] + (2.0/3.0) * (U2_i[2] + local_rhs[2] * dt)
        E_new[i,j]    = (1.0/3.0) * Un_i[3] + (2.0/3.0) * (U2_i[3] + local_rhs[3] * dt)

# ============================================
# UTILITY FUNCTIONS
# ============================================
def get_numpy_state(rho_wp, rhou_wp, rhov_wp, E_wp):
    return np.stack([rho_wp.numpy(), rhou_wp.numpy(), rhov_wp.numpy(), E_wp.numpy()], axis=0)

def exact_sod(x, t, gamma=1.4):
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1
    def f(p, rho, p_i):
        A = 2 / ((gamma + 1) * rho)
        B = (gamma - 1) / (gamma + 1) * p_i
        if p > p_i:
            return (p - p_i) * np.sqrt(A / (p + B))
        else:
            return (2 * np.sqrt(gamma * p_i / rho) / (gamma - 1)) * ((p / p_i)**((gamma - 1)/(2*gamma)) - 1)
    p = 0.5 * (pL + pR)
    for _ in range(50):
        fL, fR = f(p, rhoL, pL), f(p, rhoR, pR)
        func = fL + fR + uR - uL
        dp = 1e-6
        df = (f(p+dp, rhoL, pL) + f(p+dp, rhoR, pR) - fL - fR) / dp
        p -= func / df
        p = max(p, 1e-6)
    p_star, u_star = p, 0.5 * (uL + uR + f(p, rhoR, pR) - f(p, rhoL, pL))
    cL, cR = np.sqrt(gamma * pL / rhoL), np.sqrt(gamma * pR / rhoR)
    SHL = uL - cL
    STL = u_star - np.sqrt(gamma * p_star / (rhoL * (p_star / pL)**(1/gamma)))
    SR = uR + cR * np.sqrt((gamma + 1)/(2*gamma) * (p_star/pR - 1) + 1)
    rho, u, p_out = np.zeros_like(x), np.zeros_like(x), np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i] - 0.5) / t
        if xi < SHL: rho[i], u[i], p_out[i] = rhoL, uL, pL
        elif xi <= STL:
            u_i = (2/(gamma+1)) * (cL + xi)
            c_i = cL - (gamma-1)/2 * (u_i - uL)
            rho[i], u[i], p_out[i] = rhoL * (c_i/cL)**(2/(gamma-1)), u_i, pL * (c_i/cL)**(2*gamma/(gamma-1))
        elif xi < u_star: rho[i], u[i], p_out[i] = rhoL * (p_star/pL)**(1/gamma), u_star, p_star
        elif xi <= SR: rho[i], u[i], p_out[i] = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1) * p_star/pR + 1)), u_star, p_star
        else: rho[i], u[i], p_out[i] = rhoR, uR, pR
    return rho, u, p_out

def generate_plot_filename(base_name, scheme, grid_size, ext='png'):
    return f"{base_name}_{scheme}_opti8_fixed_{grid_size[0]}x{grid_size[1]}.{ext}"

def plot_final_results(U, t_final, gamma, NG, NX, NY):
    grid_size = (NX, NY)
    scheme_name = NUMERICAL_SCHEME.upper()
    rho = U[0, NG:-NG, NG:-NG]
    u = U[1, NG:-NG, NG:-NG] / (rho + 1e-10)
    v = U[2, NG:-NG, NG:-NG] / (rho + 1e-10)
    p = (gamma-1)*(U[3, NG:-NG, NG:-NG] - 0.5*rho*(u*u + v*v))
    p = np.maximum(p, 1e-6)
    c = np.sqrt(gamma * p / (rho + 1e-10))
    Mach = np.sqrt(u*u + v*v) / (c + 1e-10)
    grad_x = np.gradient(rho, axis=0)
    grad_y = np.gradient(rho, axis=1)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    schlieren = np.log10(grad_mag / (np.max(grad_mag) + 1e-10) + 1e-10)
    x_phys = np.linspace(0, 1, NX)
    y_phys = np.linspace(0, 1, NY)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    im1 = axes[0,0].imshow(rho.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,0].set_title(f'Density at t={t_final:.3f}')
    axes[0,0].set_xlabel('x'); axes[0,0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0,0])
    im2 = axes[0,1].imshow(p.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,1].set_title(f'Pressure at t={t_final:.3f}')
    axes[0,1].set_xlabel('x'); axes[0,1].set_ylabel('y')
    plt.colorbar(im2, ax=axes[0,1])
    vel_mag = np.sqrt(u**2 + v**2)
    im3 = axes[0,2].imshow(vel_mag.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[0,2].set_title(f'Velocity Magnitude at t={t_final:.3f}')
    axes[0,2].set_xlabel('x'); axes[0,2].set_ylabel('y')
    plt.colorbar(im3, ax=axes[0,2])
    im4 = axes[1,0].imshow(Mach.T, origin='lower', cmap='jet', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,0].set_title(f'Mach Number at t={t_final:.3f}')
    axes[1,0].set_xlabel('x'); axes[1,0].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1,0])
    im5 = axes[1,1].imshow(schlieren.T, origin='lower', cmap='gray', extent=[0, 1, 0, 1], aspect='auto')
    axes[1,1].set_title('Schlieren (log scale)')
    axes[1,1].set_xlabel('x'); axes[1,1].set_ylabel('y')
    plt.colorbar(im5, ax=axes[1,1])
    stride = max(1, min(NX, NY) // 50)
    X, Y = np.meshgrid(x_phys[::stride], y_phys[::stride])
    U_ds = u[::stride, ::stride]
    V_ds = v[::stride, ::stride]
    axes[1,2].imshow(rho.T, origin='lower', cmap='jet', alpha=0.6, extent=[0, 1, 0, 1], aspect='auto')
    axes[1,2].quiver(X, Y, U_ds.T, V_ds.T, alpha=0.8, color='white', scale=50)
    axes[1,2].set_title('Density with Velocity Vectors')
    axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('y')
    plt.suptitle(f'2D Riemann Problem - {scheme_name} (opti8_fixed), t={t_final:.3f}')
    plt.tight_layout()
    filename = generate_plot_filename('final_results', scheme_name, grid_size)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✓ {filename}")
    j_mid = NY // 2
    rho_mid = U[0, NG:-NG, j_mid]
    u_mid = U[1, NG:-NG, j_mid] / (rho_mid + 1e-10)
    p_mid = (gamma-1)*(U[3, NG:-NG, j_mid] - 0.5*rho_mid*u_mid**2)
    x = np.linspace(0, 1, NX)
    plt.figure(figsize=(12, 10))
    plt.subplot(3, 1, 1)
    plt.plot(x, rho_mid, 'b-', linewidth=2, label=f'{scheme_name} (GPU)')
    plt.ylabel('Density'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(f'Mid-plane (y=0.5) at t={t_final:.3f}')
    plt.subplot(3, 1, 2)
    plt.plot(x, u_mid, 'b-', linewidth=2)
    plt.ylabel('Velocity'); plt.grid(True, alpha=0.3)
    plt.subplot(3, 1, 3)
    plt.plot(x, p_mid, 'b-', linewidth=2)
    plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    midplane_filename = generate_plot_filename('midplane_profiles', scheme_name, grid_size)
    plt.savefig(midplane_filename, dpi=300, bbox_inches='tight')
    print(f"✓ {midplane_filename}")

# ============================================
# MAIN SIMULATION LOOP
# ============================================
print(f"Grid: {NX}×{NY}")
print(f"Scheme: {NUMERICAL_SCHEME.upper()} (GPU - {device})")
print(f"Time stepping: {TIME_STEPPING.upper()}")
print("="*50)

# Allocate SoA arrays
rho_n = wp.array(rho_np, dtype=wp.float32, device=device)
rhou_n = wp.array(rhou_np, dtype=wp.float32, device=device)
rhov_n = wp.array(rhov_np, dtype=wp.float32, device=device)
E_n = wp.array(E_np, dtype=wp.float32, device=device)

rho_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_new = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_1 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

rho_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhou_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
rhov_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)
E_2 = wp.zeros((Nx, Ny), dtype=wp.float32, device=device)

max_speed_arr = wp.zeros(1, dtype=float, device=device)

# Compute fixed dt
max_speed_arr.fill_(0.0)
wp.launch(compute_max_speed_kernel, dim=(Nx, Ny),
          inputs=[rho_n, rhou_n, rhov_n, E_n, max_speed_arr, gamma, EPS, NG, Nx, Ny],
          device=device)
wp.synchronize()
initial_max_speed = max_speed_arr.numpy()[0]
SAFETY_FACTOR = 0.5
dt_fixed = (CFL * SAFETY_FACTOR) * min(dx, dy) / initial_max_speed
print(f"Initial max speed: {initial_max_speed:.3f}")
print(f"Fixed dt = {dt_fixed:.6f}")
print(f"Estimated steps: {int(t_final/dt_fixed) + 1}")

run_times = []

for run in range(TOTAL_RUNS):
    wp.copy(rho_n, wp.array(rho_np, dtype=wp.float32, device=device))
    wp.copy(rhou_n, wp.array(rhou_np, dtype=wp.float32, device=device))
    wp.copy(rhov_n, wp.array(rhov_np, dtype=wp.float32, device=device))
    wp.copy(E_n, wp.array(E_np, dtype=wp.float32, device=device))
    wp.synchronize()

    t = 0.0
    step = 0
    start_time = time.time()

    while t < t_final:
        if IS_PROFILING and step >= 3:
            break

        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n, NG, Nx, Ny], device=device)

        dt = dt_fixed
        if t + dt > t_final:
            dt = t_final - t

        wp.launch(rk3_stage1_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_1, rhou_1, rhov_1, E_1, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage2_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_1, rhou_1, rhov_1, E_1,
                          rho_2, rhou_2, rhov_2, E_2,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)
        wp.launch(apply_bc_kernel, dim=(Nx, Ny),
                  inputs=[rho_2, rhou_2, rhov_2, E_2, NG, Nx, Ny], device=device)

        wp.launch(rk3_stage3_kernel, dim=(Nx, Ny),
                  inputs=[rho_n, rhou_n, rhov_n, E_n,
                          rho_2, rhou_2, rhov_2, E_2,
                          rho_new, rhou_new, rhov_new, E_new,
                          gamma, EPS, inv_dx, inv_dy, float(dt), NG, Nx, Ny], device=device)

        wp.copy(rho_n, rho_new)
        wp.copy(rhou_n, rhou_new)
        wp.copy(rhov_n, rhov_new)
        wp.copy(E_n, E_new)

        t += dt
        step += 1

    end_time = time.time()
    elapsed = end_time - start_time
    run_times.append(elapsed)
    print(f"Run {run+1}/{TOTAL_RUNS} completed in {elapsed:.4f} s")

print("\n" + "="*50)
if not IS_PROFILING:
    if TOTAL_RUNS > 1:
        valid_times = run_times[1:]
        best_time = min(valid_times)
        print(f"Discarding warm-up: {run_times[0]:.4f} s")
        print(f"Best time (runs 2-{TOTAL_RUNS}): {best_time:.4f} s")
    else:
        best_time = run_times[0]
        print(f"Execution time: {best_time:.4f} s")

    flops_per_cell_per_step = 12000
    total_cells = NX * NY
    total_flops = total_cells * step * flops_per_cell_per_step
    tflops = total_flops / (best_time * 1e12)
    print(f"Total steps: {step}")
    print(f"Achieved TFLOPS: {tflops:.6f} TFLOPS")
else:
    print("Profiling mode – ignoring timing.")
print("="*50)

print("\n" + "="*50)
print("CREATING FINAL PLOTS")
U_final = get_numpy_state(rho_n, rhou_n, rhov_n, E_n)
plot_final_results(U_final, t_final, gamma, NG, NX, NY)

print("\nVALIDATION WITH EXACT SOD SOLUTION")
j_mid = NY // 2
rho_num = U_final[0, NG:-NG, j_mid]
u_num = U_final[1, NG:-NG, j_mid] / (rho_num + 1e-10)
p_num = (gamma-1)*(U_final[3, NG:-NG, j_mid] - 0.5*rho_num*u_num**2)
x = np.linspace(0, 1, NX)
rho_ex, u_ex, p_ex = exact_sod(x, t_final, gamma)
print(f"L1 errors at t={t_final}:")
print(f"  Density:  {np.mean(np.abs(rho_num - rho_ex)):.6f}")
print(f"  Velocity: {np.mean(np.abs(u_num - u_ex)):.6f}")
print(f"  Pressure: {np.mean(np.abs(p_num - p_ex)):.6f}")

plt.figure(figsize=(12,10))
plt.subplot(3,1,1)
plt.plot(x, rho_num, 'b-', label=f'{NUMERICAL_SCHEME.upper()} (GPU)')
plt.plot(x, rho_ex, 'r--', label='Exact')
plt.ylabel('Density'); plt.legend(); plt.grid(True)
plt.title(f'Sod shock tube validation, t={t_final}')
plt.subplot(3,1,2)
plt.plot(x, u_num, 'b-'); plt.plot(x, u_ex, 'r--')
plt.ylabel('Velocity'); plt.grid(True)
plt.subplot(3,1,3)
plt.plot(x, p_num, 'b-'); plt.plot(x, p_ex, 'r--')
plt.ylabel('Pressure'); plt.xlabel('x'); plt.grid(True)
plt.tight_layout()
validation_filename = generate_plot_filename('validation', NUMERICAL_SCHEME, (NX, NY))
plt.savefig(validation_filename, dpi=300, bbox_inches='tight')
print(f"✓ {validation_filename}\nAll plots generated.")